# Сравнение детекторов аномалий траектории



In [ ]:
TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu124"
ALLOWED_STATUS = ("move", "anomaly")

TEST_SPLIT_1 = 0.6
TEST_SPLIT_2 = 0.8
MCC_BAND = 0.02
PLOT_TAIL = 50_000
MAP_FIGSIZE = (22.0, 11.0)
N_WINDOWS = 8_000
N_EPOCHS = 10
TORCH_BATCH_SIZE = 256
TORCH_INFER_BATCH = 512
TORCH_LR = 1e-3

## Чтение и предобработка данных

Исходный файл csv задаёт последовательность записей с полями времени, координат и статуса. После чтения строится очищенный ряд наблюдений и две пары выборок «обучение / тест» с разными долями обучения.

После загрузки CSV имеем индексы $j=0,\ldots,M-1$. Сырые поля: $t_j$ (time), $\varphi_j$ (lat), $\lambda_j$ (lon), $y_j$ (status). Допустимые значения метки: $y_j\in\{\mathrm{move},\mathrm{anomaly},\ldots\}$; в эксперименте остаются только $\mathrm{move}$ и $\mathrm{anomaly}$. Для метрик классификации положительный класс — аномалия ($y_i=\mathrm{anomaly}$); в столбце label она кодируется нулём, штатный ход — единицей.

1. Отбор столбцов. Оставляем только
$$(t_i,\varphi_i,\lambda_i,y_i)_{i=0}^{N-1},$$
где $t_i$ — время, $\varphi_i$ — широта, $\lambda_i$ — долгота, $y_i$ — метка класса.

2. Валидные координаты. Исключаются пропуски, нечисловые координаты, выход за допустимые пределы WGS-84 и точки «нулевого» GPS:
$$\mathcal{J}_1=\{i:\ \varphi_i,\lambda_i\ \text{конечны},\ -90\le\varphi_i\le 90,\ -180\le\lambda_i\le 180,\ (\varphi_i,\lambda_i)\neq(0,0)\}.$$

3. Бинарная задача. Остаются только штатный ход и аномалия:
$$\mathcal{J}_2=\{i\in\mathcal{J}_1:\ y_i\in\{\mathrm{move},\mathrm{anomaly}\}\}.$$

4. Упорядочивание. На $\mathcal{J}_2$ задаётся сортировка по возрастанию $t_i$. Индекс $i=0$ — первая валидная точка ряда.

5. Локальная ENU (м). Опорная точка — первая валидная точка: $(\varphi_0,\lambda_0)=(\varphi_{0},\lambda_{0})$ в нумерации после шагов 1–4. Координаты WGS-84 переводятся в плоскость «восток–север» (WGS-84 $\to$ ECEF $\to$ ENU):
$$(x_i,n_i)\equiv(E_i,N_i)=\mathrm{ENU}(\varphi_i,\lambda_i\mid\varphi_0,\lambda_0).$$
Поля $\varphi_i,\lambda_i$ сохраняются; столбцы $x_i$, $n_i$ добавляются в таблицу (метка класса по-прежнему $y_i$).

6. Обрезка по крайним $\mathrm{move}$. Пусть
$$\mathcal{I}_{\mathrm{move}}=\{i:\ y_i=\mathrm{move}\},\quad i_{\min}=\min\mathcal{I}_{\mathrm{move}},\quad i_{\max}=\max\mathcal{I}_{\mathrm{move}}.$$
Рабочий ряд — срез $i_{\min}\le i\le i_{\max}$ (первая и последняя точки ряда — $\mathrm{move}$); координаты $(x_i,n_i)$ не пересчитываются.

7. Разбиение train / test. Для доли обучения $\rho\in(0,1)$ (в блокноте $\rho=\texttt{TEST\_SPLIT\_1}$ или $\texttt{TEST\_SPLIT\_2}$) при длине ряда $N$ задаётся пороговый индекс
$$k_0=\lfloor\rho N\rfloor.$$
Граница разбиения — первый индекс штатного хода не левее $k_0$:
$$k=\min\{i\ge k_0:\ y_i=\mathrm{move}\}.$$
Если $y_{k_0}=\mathrm{move}$, то $k=k_0$. Обучение: индексы $[0,k)$, тест: $[k,N)$. Так граница не попадает внутрь чисто аномального фрагмента без соседнего $\mathrm{move}$.



In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
import torch
from IPython.display import display
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import PyroOptim
from torch.distributions import Normal, kl_divergence
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from app.evaluation import GeospatialMetrics, TrajectoryMetrics, TrajectorySeries
from app.evaluation.classification_metrics import ClassificationMetrics
from numpy.linalg import LinAlgError
from scipy.linalg import solve_triangular
from app.geo.local_enu import LocalEnuConverter
from app.viz import plot_track_maps
from statsmodels.tsa.arima.model import ARIMA
from hmmlearn.hmm import GaussianHMM

path_root = Path.cwd().parent
path_data = path_root / "data" / "2.csv"

get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Корень: {path_root}")
print(f"Файл: {path_data}")
print(f"PyTorch {torch.__version__}, устройство {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

## MCC


In [ ]:
def nfmt(value: int) -> str:
    """Форматирует целое число с разделителем тысяч (пробел).

    Args:
        value: целое значение для вывода.

    Returns:
        str: строка вида «1 234 567».
    """
    return f"{int(value):,}".replace(",", " ")


def fmt_num(value: float, decimals: int = 2) -> str:
    """Число; при |value| >= 1000 — разделитель тысяч «_».

    Args:
        value: Значение для вывода.
        decimals: Число знаков после запятой (0 — целое).

    Returns:
        str: Отформатированная строка или «—» для нечисловых значений.
    """
    if value is None or not np.isfinite(float(value)):
        return "—"
    number = float(value)
    negative = number < 0
    number = abs(number)
    if number < 1000:
        text = f"{number:.{decimals}f}" if decimals else str(int(round(number)))
        if "." in text:
            text = text.rstrip("0").rstrip(".")
        return ("-" if negative else "") + text
    if decimals == 0:
        body = f"{int(round(number)):,}".replace(",", "_")
    else:
        whole = int(number)
        fraction = number - whole
        fraction_text = f"{fraction:.{decimals}f}".split(".")[1].rstrip("0")
        body = f"{whole:,}".replace(",", "_")
        if fraction_text:
            body = f"{body}.{fraction_text}"
    return ("-" if negative else "") + body


def select_trajectory_columns(raw: pd.DataFrame) -> pd.DataFrame:
    """Оставляет столбцы time, lat, lon и переименовывает status в y.

    Args:
        raw: таблица из data/2.csv.

    Returns:
        pd.DataFrame: столбцы time, lat, lon, y.
    """
    return raw.loc[:, ["time", "lat", "lon", "status"]].rename(columns={"status": "y"}).copy()


def coerce_coordinate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Приводит lat и lon к числовому типу (нечисловые значения → NaN).

    Args:
        frame: таблица с столбцами lat, lon.

    Returns:
        pd.DataFrame: копия с числовыми lat, lon.
    """
    result = frame.copy()
    result["lat"] = pd.to_numeric(result["lat"], errors="coerce")
    result["lon"] = pd.to_numeric(result["lon"], errors="coerce")
    return result


def filter_finite_coordinates(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с пропусками в lat или lon.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: только строки с конечными координатами.
    """
    mask = frame["lat"].notna() & frame["lon"].notna()
    return frame.loc[mask].reset_index(drop=True)


def filter_coordinates_wgs84(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с координатами вне допустимых пределов WGS-84.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: lat в [-90, 90], lon в [-180, 180].
    """
    mask = frame["lat"].between(-90.0, 90.0) & frame["lon"].between(-180.0, 180.0)
    return frame.loc[mask].reset_index(drop=True)


def filter_nonzero_gps(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет точки «нулевого» GPS (lat, lon) = (0, 0).

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    mask = ~((frame["lat"] == 0.0) & (frame["lon"] == 0.0))
    return frame.loc[mask].reset_index(drop=True)


def filter_allowed_status(frame: pd.DataFrame, allowed: tuple[str, ...]) -> pd.DataFrame:
    """Оставляет только метки из allowed (move, anomaly).

    Args:
        frame: таблица со столбцом y.
        allowed: допустимые строковые значения status.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    labels = frame["y"].astype(str)
    return frame.loc[labels.isin(allowed)].reset_index(drop=True)


def sort_by_time(frame: pd.DataFrame) -> pd.DataFrame:
    """Сортирует ряд по возрастанию time (стабильная сортировка).

    Args:
        frame: таблица со столбцом time.

    Returns:
        pd.DataFrame: упорядоченный ряд.
    """
    order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
    return frame.iloc[order].reset_index(drop=True)


def attach_local_enu(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбцы x, n — локальная ENU (м) от первой точки ряда.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: с добавленными x, n.
    """
    result = frame.copy()
    enu = LocalEnuConverter.from_first_point(result["lat"].iloc[0], result["lon"].iloc[0])
    east, north = enu.to_local_xy(result["lat"].to_numpy(), result["lon"].to_numpy())
    result["x"] = east
    result["n"] = north
    return result


def trim_to_extreme_move(frame: pd.DataFrame) -> pd.DataFrame:
    """Обрезает ряд от первой до последней точки со статусом move.

    Args:
        frame: таблица со столбцом y.

    Returns:
        pd.DataFrame: срез между крайними move.

    Raises:
        RuntimeError: если в ряду нет точек move.
    """
    is_move = frame["y"].astype(str).to_numpy() == "move"
    if not np.any(is_move):
        raise RuntimeError("После фильтрации нет точек move.")
    move_idx = np.flatnonzero(is_move)
    return frame.iloc[move_idx[0] : move_idx[-1] + 1].reset_index(drop=True)


def attach_binary_label(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбец label для TrajectoryMetrics / TrajectorySeries.

    Положительный класс в метриках (MCC, precision, recall) — anomaly. В этом
    проекте он кодируется нулём (не единицей): anomaly → 0, move → 1. См.
    ClassificationMetrics и anomaly_mask (label == 0).

    Args:
        frame: таблица со столбцом y (строки move / anomaly).

    Returns:
        pd.DataFrame: с столбцом label (float).
    """
    result = frame.copy()
    result["label"] = np.where(result["y"].astype(str) == "anomaly", 0.0, 1.0)
    return result


def split_train_test_move(frame: pd.DataFrame, train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит ряд на обучение [0, k) и тест [k, N); граница k — первый move с индексом i >= floor(rho*N).

    Args:
        frame: предобработанный ряд с столбцом y (move / anomaly).
        train_fraction: доля обучения rho из (0, 1), как TEST_SPLIT_1 или TEST_SPLIT_2.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (train, test).

    Raises:
        ValueError: если train_fraction не в (0, 1).
        RuntimeError: если ряд слишком короткий, нет подходящего move или выборка пуста.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction должен быть в (0, 1).")
    n = len(frame)
    if n < 2:
        raise RuntimeError("Слишком короткий ряд для разбиения.")
    k0 = int(np.floor(train_fraction * n))
    k0 = min(max(k0, 1), n - 1)
    labels = frame["y"].astype(str).to_numpy()
    move_from_k0 = np.flatnonzero((np.arange(n) >= k0) & (labels == "move"))
    if move_from_k0.size == 0:
        raise RuntimeError(f"Нет move для границы при rho={train_fraction}.")
    k = int(move_from_k0[0])
    train = frame.iloc[:k].reset_index(drop=True)
    test = frame.iloc[k:].reset_index(drop=True)
    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(f"Пустая выборка при rho={train_fraction}, k={k}, N={n}.")
    return train, test


def describe_split(name: str, train: pd.DataFrame, test: pd.DataFrame) -> None:
    """Печатает размер и доли классов anomaly и move для обучения и теста.

    Args:
        name: заголовок сценария (например, df1 и доля rho).
        train: обучающая выборка.
        test: тестовая выборка.
    """
    for part_name, part in (("обучение", train), ("тест", test)):
        labels = part["y"].astype(str)
        n = len(part)
        p_anom = float((labels == "anomaly").mean())
        p_move = float((labels == "move").mean())
        print(
            f"{name}: {part_name:<9} {nfmt(n):>10} точек   "
            f"anomaly {p_anom:6.1%}   move {p_move:6.1%}"
        )


raw_full = pd.read_csv(path_data)
frame = select_trajectory_columns(raw_full)
frame = coerce_coordinate_columns(frame)
frame = filter_finite_coordinates(frame)
frame = filter_coordinates_wgs84(frame)
frame = filter_nonzero_gps(frame)
frame = filter_allowed_status(frame, ALLOWED_STATUS)
frame = sort_by_time(frame)
frame = attach_local_enu(frame)
frame = trim_to_extreme_move(frame)
frame = attach_binary_label(frame)
train_1, test_1 = split_train_test_move(frame, TEST_SPLIT_1)
train_2, test_2 = split_train_test_move(frame, TEST_SPLIT_2)

df1 = {"train": train_1, "test": test_1}
df2 = {"train": train_2, "test": test_2}

print(f"Исходный CSV: {nfmt(len(raw_full))} строк, после предобработки: {nfmt(len(frame))}.")
describe_split("df1 (rho={:.0%})".format(TEST_SPLIT_1), train_1, test_1)
describe_split("df2 (rho={:.0%})".format(TEST_SPLIT_2), train_2, test_2)

## Используемые метрики

### Коэффициент корреляции Мэттьюса (MCC)

Коэффициент корреляции Мэттьюса (Matthews correlation coefficient, MCC) — коэффициент корреляции Пирсона между двумя бинарными векторами: эталонной разметкой и предсказанием (каждая компонента кодируется как 0 или 1). Эквивалентно он выражается через четыре числа таблицы сопряжённости $TP$, $FP$, $FN$, $TN$ и принимает значения от $-1$ до $1$.

Используется в задачах бинарной классификации, в том числе при сильном дисбалансе классов: оценка предсказаний структуры белка, медицинская диагностика, обнаружение аномалий, сравнение порогов и моделей, когда accuracy и доля «большого» класса малоинформативны.

Источник: Chicco D., Jurman G. *The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation* // BMC Genomics. 2020. Vol. 21, Art. 6. Open access: [10.1186/s12864-019-6413-7](https://link.springer.com/article/10.1186/s12864-019-6413-7). (В статье отмечено, что MCC изначально предложен B. W. Matthews в 1975 г.; в статье приведены математические свойства и сравнение с accuracy и $F_1$.)

Основное преимущество: метрика одновременно учитывает все четыре ячейки таблицы сопряжённости и симметрично отражает цену ложных срабатываний и пропусков; при дисбалансе классов тривиальные предсказания «всегда отрицательный» или «всегда положительный» класс обычно дают MCC около нуля, тогда как accuracy может оставаться высокой.

Для положительного класса (в этом блокноте положительный класс — **anomaly**, $y_i=0$, бинарная метка $a_i=\mathbf{1}[y_i=0]$; предсказание $\hat a_i=\mathbf{1}[\hat y_i=0]$):

$$
\mathrm{MCC}=\frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}},
$$
где:\
$TP=\sum_i \mathbf{1}[a_i=1\land \hat a_i=1]$,\
$FP=\sum_i \mathbf{1}[a_i=0\land \hat a_i=1]$,\
$FN=\sum_i \mathbf{1}[a_i=1\land \hat a_i=0]$,\
$TN=\sum_i \mathbf{1}[a_i=0\land \hat a_i=0]$.

Интерпретация: $\mathrm{MCC}\in[-1,1]$; $1$ — полное совпадение масок; $0$ — согласие на уровне случайного предсказания при тех же маргинальных долях классов; отрицательные значения — систематическое расхождение хуже случайного. Если знаменатель равен нулю (например, в выборке или в предсказании только один класс), MCC в строгом смысле не определён; в коде этого блокнота в таком случае принимается $0$.

### Distance loss

Геопространственная метрика: насколько расходятся длины эталонной и предсказанной траектории (в метрах). Длина ломаной — сумма геодезических звеньев $d_{\mathrm{geo}}$; distance loss — модуль разности:

$$
L(T)=\sum_i d_{\mathrm{geo}}(p_i,p_{i+1}), \qquad DL=\bigl|L_{\mathrm{pred}}-L_{\mathrm{ref}}\bigr|.
$$

$L_{\mathrm{ref}}$ строится по $G$ (штатные точки и линейная интерполяция в аномалиях между соседними штатными). $L_{\mathrm{pred}}$ — по $P$: в ломаную входят только точки, принятые за штатный ход ($\hat y_i\neq 0$); пропуски не интерполируются, соседние принятые точки соединяются напрямую. На тесте $DL$ также считается через на заполненных траекториях; при выборе порога по MCC в полосе близкого значения MCC минимизируется операционный distance loss (длина по принятым move-точкам).

In [ ]:
def trajectory_series_pair(frame: pd.DataFrame, predicted_anomaly: np.ndarray) -> tuple[TrajectorySeries, TrajectorySeries]:
    """Собирает prediction и ground truth для TrajectoryMetrics.

    Args:
        frame: Строки test с полями time, lat, lon, y.
        predicted_anomaly: True, если модель объявила anomaly.

    Returns:
        tuple[TrajectorySeries, TrajectorySeries]: (prediction, ground_truth).
    """
    ground_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    truth = np.where(ground_anomaly, 0.0, 1.0)
    predicted = np.where(predicted_anomaly, 0.0, 1.0)
    time = frame["time"].to_numpy()
    lon = frame["lon"].to_numpy(dtype=float)
    lat = frame["lat"].to_numpy(dtype=float)
    ground_truth = TrajectorySeries(time, lon, lat, truth)
    prediction = TrajectorySeries(time, lon, lat, predicted)
    return prediction, ground_truth


def slice_trajectory_series(series: TrajectorySeries, tail: int) -> TrajectorySeries:
    """Хвост TrajectorySeries для карты (последние tail точек).

    Args:
        series: Исходный ряд.
        tail: Число точек с конца.

    Returns:
        TrajectorySeries: Срез хвоста.
    """
    count = min(int(tail), len(series.time))
    return TrajectorySeries(
        series.time[-count:],
        series.lon[-count:],
        series.lat[-count:],
        series.label[-count:],
    )


def evaluate_test(frame: pd.DataFrame, predicted_anomaly: np.ndarray) -> dict[str, float]:
    """Считает на тесте метрики из Отчёта (TrajectoryMetrics.compute_all).

    Args:
        frame: Тестовый DataFrame.
        predicted_anomaly: Маска предсказанных anomaly.

    Returns:
        dict[str, float]: Гео-, временные и классификационные метрики test.
    """
    prediction, ground_truth = trajectory_series_pair(frame, predicted_anomaly)
    return TrajectoryMetrics.compute_all(prediction, ground_truth)


def time_seconds_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Время ряда в секундах от эпохи (float64).

    Args:
        frame: DataFrame с колонкой time, приводимой к datetime.

    Returns:
        np.ndarray: Секунды от эпохи Unix, shape (N,).
    """
    stamp = pd.to_datetime(frame["time"])
    return stamp.to_numpy(dtype="datetime64[ns]").astype(np.int64) / 1e9


def enu_step_components(east_m: np.ndarray, north_m: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Приращения ENU; у индекса 0 шаг 0.

    Args:
        east_m: Восточная координата ENU, м.
        north_m: Северная координата ENU, м.

    Returns:
        tuple[np.ndarray, np.ndarray]: (dx, dn), м; dx[0] = dn[0] = 0.
    """
    east = np.asarray(east_m, dtype=np.float64)
    north = np.asarray(north_m, dtype=np.float64)
    dx = np.zeros(len(east), dtype=np.float64)
    dn = np.zeros(len(north), dtype=np.float64)
    if len(east) > 1:
        dx[1:] = np.diff(east)
        dn[1:] = np.diff(north)
    return dx, dn


def enu_xy_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Координаты ENU (восток, север), м, форма (N, 2).

    Args:
        frame: DataFrame с колонками x (восток) и n (север), м.

    Returns:
        np.ndarray: Матрица (N, 2) координат ENU, м.
    """
    return np.column_stack([frame["x"].to_numpy(dtype=float), frame["n"].to_numpy(dtype=float)])


def incoming_velocity_enu(
    east_m: np.ndarray,
    north_m: np.ndarray,
    time_sec: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Скорость по входящему шагу; у i=0 нули. step_dt[0]=1.

    Args:
        east_m: Восточная координата ENU, м.
        north_m: Северная координата ENU, м.
        time_sec: Время точек, с.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: (v_E, v_N, step_dt), м/с и с.
    """
    xy = np.column_stack([np.asarray(east_m, float), np.asarray(north_m, float)])
    t_sec = np.asarray(time_sec, dtype=np.float64)
    count = len(xy)
    velocity_east = np.zeros(count)
    velocity_north = np.zeros(count)
    step_dt = np.ones(count)
    if count < 2:
        return velocity_east, velocity_north, step_dt
    step_dt[1:] = np.diff(t_sec)
    safe_dt = np.clip(step_dt[1:], 1e-3, None)
    velocity_east[1:] = np.diff(xy[:, 0]) / safe_dt
    velocity_north[1:] = np.diff(xy[:, 1]) / safe_dt
    return velocity_east, velocity_north, step_dt


def wrap_angle(radians: np.ndarray) -> np.ndarray:
    """Перенос угла в интервал (-pi, pi].

    Args:
        radians: Углы, рад.

    Returns:
        np.ndarray: Те же углы в интервале (-pi, pi].
    """
    return (np.asarray(radians, dtype=float) + np.pi) % (2.0 * np.pi) - np.pi


def trajectory_kinematics(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Скорость, ускорение, поворот курса и Δt; у i=0 нули.

    Args:
        frame: DataFrame с колонками x, n и time.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: speed (м/с), acc (м/с²),
        dpsi (рад) и step_dt (с).
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    time_sec = time_seconds_from_frame(frame)
    count = len(east)
    step_dt = np.ones(count, dtype=float)
    speed = np.zeros(count, dtype=float)
    acc = np.zeros(count, dtype=float)
    dpsi = np.zeros(count, dtype=float)
    if count < 2:
        return speed, acc, dpsi, step_dt
    step_dt[1:] = np.diff(time_sec)
    dist = np.hypot(np.diff(east), np.diff(north))
    speed[1:] = dist / np.clip(step_dt[1:], 1.0, None)
    acc[1:] = np.diff(speed) / np.clip(step_dt[1:], 1.0, None)
    heading = np.zeros(count, dtype=float)
    dx_step = east[1:] - east[:-1]
    dy_step = north[1:] - north[:-1]
    heading[1:] = np.arctan2(dx_step, dy_step)
    if count > 2:
        dpsi[2:] = wrap_angle(np.diff(heading[1:]))
    return speed, acc, dpsi, step_dt


def robust_params(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Робастная шкала: медиана и межквартильный размах по столбцам.

    Args:
        matrix: Матрица признаков shape (n, d).

    Returns:
        tuple[np.ndarray, np.ndarray]: center (медианы) и scale (IQR); scale < 1e-6 → 1.
    """
    center = np.median(matrix, axis=0)
    scale = np.subtract(*np.percentile(matrix, [75, 25], axis=0))
    scale = np.where(scale < 1e-6, 1.0, scale)
    return center, scale


def apply_robust(matrix: np.ndarray, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Центрирование и деление на scale.

    Args:
        matrix: Матрица признаков shape (n, d).
        center: Центры столбцов из robust_params.
        scale: Масштабы столбцов из robust_params.

    Returns:
        np.ndarray: (matrix - center) / scale.
    """
    return (matrix - center) / scale


def track_shape_index(
    east_m: np.ndarray, north_m: np.ndarray, step_dist: np.ndarray, window: int
) -> np.ndarray:
    """Отношение длины пути к смещению за окно (прямой ход ≈ 1).

    Args:
        east_m: Восточная координата ENU, м.
        north_m: Северная координата ENU, м.
        step_dist: Длины шагов, м.
        window: Размер окна в точках.

    Returns:
        np.ndarray: shape той же длины; первые window значений равны 1.
    """
    count = len(step_dist)
    shape = np.ones(count, dtype=float)
    if count <= window:
        return shape
    cumulative = np.cumsum(step_dist)
    path = cumulative[window:] - cumulative[:-window]
    span = np.hypot(east_m[window:] - east_m[:-window], north_m[window:] - north_m[:-window])
    shape[window:] = path / np.maximum(span, 1.0)
    return shape


def dbscan_density_matrix(frame: pd.DataFrame, shape_window: int) -> np.ndarray:
    """Признаки плотности: Δx, Δy, v, a, Δψ, log(shape); без абсолютных координат.

    Args:
        frame: DataFrame с колонками x, n и time.
        shape_window: Окно для track_shape_index.

    Returns:
        np.ndarray: Матрица (N, 6) признаков плотности.
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    speed, acc, dpsi, _ = trajectory_kinematics(frame)
    step_dx = np.zeros(len(east), dtype=float)
    step_dy = np.zeros(len(east), dtype=float)
    step_dist = np.zeros(len(east), dtype=float)
    if len(east) > 1:
        step_dx[1:] = np.diff(east)
        step_dy[1:] = np.diff(north)
        step_dist[1:] = np.hypot(step_dx[1:], step_dy[1:])
    shape = track_shape_index(east, north, step_dist, shape_window)
    return np.column_stack(
        [step_dx, step_dy, speed, acc, dpsi, np.log(np.maximum(shape, 1.0))]
    )


def dbscan_robust_features(
    frame: pd.DataFrame, center: np.ndarray, scale: np.ndarray, shape_window: int
) -> np.ndarray:
    """Робастно нормированная матрица признаков DBSCAN.

    Args:
        frame: DataFrame с колонками x, n и time.
        center: Центры из robust_params(train move).
        scale: Масштабы из robust_params(train move).
        shape_window: Окно для track_shape_index.

    Returns:
        np.ndarray: Матрица (N, 6) после apply_robust.
    """
    return apply_robust(dbscan_density_matrix(frame, shape_window), center, scale)


def dbscan_isolation_scores(
    features: np.ndarray, min_samples: int, core_indices: np.ndarray | None
) -> np.ndarray:
    """max(k-distance, расстояние до ближайшей core-точки); больше — изолированнее.

    Args:
        features: Нормированные признаки shape (n, d).
        min_samples: k для k-distance.
        core_indices: Индексы core-точек DBSCAN или None.

    Returns:
        np.ndarray: Score длины n.
    """
    features = np.asarray(features, dtype=float)
    neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(features)
    k_distance = neighbors.kneighbors(features)[0][:, -1]
    if core_indices is None or len(core_indices) == 0:
        return k_distance
    core_nn = NearestNeighbors(n_neighbors=1, algorithm="kd_tree").fit(features[core_indices])
    core_distance = core_nn.kneighbors(features)[0][:, 0]
    return np.maximum(k_distance, core_distance)


def isolation_forest_robust_features(
    frame: pd.DataFrame, center: np.ndarray, scale: np.ndarray, shape_window: int
) -> np.ndarray:
    """Робастно нормированные признаки IF (как у DBSCAN).

    Args:
        frame: DataFrame с колонками x, n и time.
        center: Центры из robust_params(train move).
        scale: Масштабы из robust_params(train move).
        shape_window: Окно для track_shape_index.

    Returns:
        np.ndarray: Матрица (N, 6) после apply_robust.
    """
    return apply_robust(dbscan_density_matrix(frame, shape_window), center, scale)


def frame_window_arrays(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Поля для оконных детекторов: ENU, шаги, кинематика, move.

    Args:
        frame: DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, np.ndarray]: count, xy, dx, dy, speed, dpsi, dt, dist, move.
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    speed, _, dpsi, step_dt = trajectory_kinematics(frame)
    count = len(east)
    step_dx = np.zeros(count, dtype=float)
    step_dy = np.zeros(count, dtype=float)
    step_dist = np.zeros(count, dtype=float)
    if count > 1:
        step_dx[1:] = np.diff(east)
        step_dy[1:] = np.diff(north)
        step_dist[1:] = np.hypot(step_dx[1:], step_dy[1:])
    return {
        "count": count,
        "xy": np.column_stack([east, north]),
        "dx": step_dx,
        "dy": step_dy,
        "speed": speed,
        "dpsi": dpsi,
        "dt": step_dt,
        "dist": step_dist,
        "move": frame["y"].astype(str).to_numpy() == "move",
    }


def window_starts(
    arrays: dict[str, np.ndarray],
    length: int,
    max_dt: float,
    max_dist: float,
    require_move: bool,
) -> np.ndarray:
    """Индексы окон без большого разрыва по времени или расстоянию.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        length: Длина окна в точках.
        max_dt: Максимальный Δt между соседними точками, с.
        max_dist: Максимальный шаг между соседними точками, м.
        require_move: Требовать move во всех точках окна.

    Returns:
        np.ndarray: Индексы допустимых стартов; пустой массив, если окон нет.
    """
    count = arrays["count"]
    if count < length:
        return np.empty(0, dtype=int)
    good = np.ones(count, dtype=bool)
    if require_move:
        good &= arrays["move"]
    step_ok = np.ones(count, dtype=bool)
    step_ok[1:] = (arrays["dt"][1:] <= max_dt) & (arrays["dist"][1:] <= max_dist)
    good_cum = np.cumsum(good.astype(np.int32))
    step_cum = np.cumsum(step_ok.astype(np.int32))
    starts = np.arange(count - length + 1)
    good_sum = good_cum[starts + length - 1] - np.where(starts > 0, good_cum[starts - 1], 0)
    step_sum = step_cum[starts + length - 1] - step_cum[starts]
    return starts[(good_sum == length) & (step_sum == length - 1)]


def take_window_starts(starts: np.ndarray, limit: int, seed: int) -> np.ndarray:
    """Случайная подвыборка стартов окон без повторов.

    Args:
        starts: Допустимые индексы начала окна.
        limit: Максимальное число стартов.
        seed: Seed генератора.

    Returns:
        np.ndarray: Отсортированные выбранные старты.
    """
    if len(starts) <= limit:
        return starts
    picked = np.random.default_rng(seed).choice(len(starts), size=limit, replace=False)
    return np.sort(starts[picked])


def scatter_window_max(
    count: int, starts: np.ndarray, values: np.ndarray, length: int
) -> np.ndarray:
    """Оконные score по точкам: в пересечениях — максимум.

    Args:
        count: Длина ряда.
        starts: Индексы начала окон.
        values: Score на окно shape (n_windows,) или (n_windows, length).
        length: Длина окна в точках.

    Returns:
        np.ndarray: Score длины count; точки вне окон равны 0.
    """
    scores = np.full(count, -np.inf)
    index = starts[:, None] + np.arange(length)[None, :]
    if np.ndim(values) == 1:
        payload = np.repeat(values, length)
    else:
        payload = np.asarray(values, dtype=float).ravel()
    np.maximum.at(scores, index.ravel(), payload)
    scores[~np.isfinite(scores)] = 0.0
    return scores


def split_window_starts(
    starts: np.ndarray, seed: int, train_fraction: float = 0.85
) -> tuple[np.ndarray, np.ndarray]:
    """Делит старты окон на обучение и калибровку.

    Args:
        starts: Индексы начала окон.
        seed: Seed перемешивания.
        train_fraction: Доля стартов в обучающей части.

    Returns:
        tuple[np.ndarray, np.ndarray]: (train_starts, calib_starts).
    """
    order = np.random.default_rng(seed).permutation(len(starts))
    cut = max(1, int(len(starts) * train_fraction))
    return starts[order[:cut]], starts[order[cut:]]


def relative_xy_km_windows(
    arrays: dict[str, np.ndarray], starts: np.ndarray, length: int
) -> np.ndarray:
    """Окна относительных координат в км от первой точки окна.

    Args:
        arrays: Кэш ряда (xy и др.) из frame_window_arrays.
        starts: Индексы начала окон.
        length: Длина окна в точках.

    Returns:
        np.ndarray: Форма (n_windows, length, 2), float32.
    """
    index = starts[:, None] + np.arange(length)[None, :]
    block = arrays["xy"][index]
    return ((block - block[:, :1, :]) / 1000.0).astype(np.float32)


def diagonal_mahalanobis(error: np.ndarray, mean: np.ndarray, var: np.ndarray) -> np.ndarray:
    """Диагональный квадрат Махаланобиса по столбцам ошибки.

    Args:
        error: Матрица остатков, shape (n, d).
        mean: Среднее по столбцам.
        var: Дисперсии по столбцам.

    Returns:
        np.ndarray: Score длины n.
    """
    return np.sum((error - mean) ** 2 / var, axis=1)


def train_torch_autoencoder(
    model: nn.Module,
    windows: np.ndarray,
    epochs: int,
    batch_size: int,
    lr: float,
) -> nn.Module:
    """Обучает автоэнкодер на окнах траектории (MSE, Adam).

    Args:
        model: PyTorch-модель с forward(batch) -> reconstruction.
        windows: Массив окон, shape (n, length, features).
        epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.

    Returns:
        nn.Module: Обученная model в режиме eval.
    """
    model.to(DEVICE)
    array = np.clip(np.ascontiguousarray(windows), -30.0, 30.0).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(array)), batch_size=batch_size, shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(batch) - batch) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch)
            seen += len(batch)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


def batched_model_forward(
    model: nn.Module, array: np.ndarray, forward_fn, batch_size: int
) -> np.ndarray:
    """Пакетный forward PyTorch-модели без градиентов.

    Args:
        model: Модель на DEVICE.
        array: Входной numpy-массив.
        forward_fn: Callable (model, batch_tensor) -> tensor.
        batch_size: Размер пакета.

    Returns:
        np.ndarray: Склеенные выходы по оси 0.
    """
    model.eval()
    pieces: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(array), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(array[start : start + batch_size])).to(
                DEVICE
            )
            pieces.append(forward_fn(model, batch).detach().cpu().numpy())
    return np.concatenate(pieces, axis=0)


def operational_path_length_m(east_m: np.ndarray, north_m: np.ndarray, move_mask: np.ndarray) -> float:
    """Длина ломаной в ENU только по точкам move (без интерполяции).

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        move_mask: True для штатного хода.

    Returns:
        float: Длина в метрах.
    """
    move = np.asarray(move_mask, dtype=bool)
    east = np.asarray(east_m, dtype=np.float64)[move]
    north = np.asarray(north_m, dtype=np.float64)[move]
    if east.size < 2:
        return 0.0
    return float(np.sum(np.hypot(np.diff(east), np.diff(north))))


def operational_distance_loss_m(
    east_m: np.ndarray,
    north_m: np.ndarray,
    ground_truth_anomaly: np.ndarray,
    predicted_anomaly: np.ndarray,
) -> float:
    """Операционный |L_pred - L_ref| по move-точкам в ENU.

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        ground_truth_anomaly: True для anomaly в разметке.
        predicted_anomaly: True для предсказанной anomaly.

    Returns:
        float: Distance loss в метрах.
    """
    gt_anomaly = np.asarray(ground_truth_anomaly, dtype=bool)
    pred_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    reference_m = operational_path_length_m(east_m, north_m, ~gt_anomaly)
    predicted_m = operational_path_length_m(east_m, north_m, ~pred_anomaly)
    return float(abs(predicted_m - reference_m))


def _threshold_record(
    tau: float,
    predicted_anomaly: np.ndarray,
    frame: pd.DataFrame,
    ground_truth_anomaly: np.ndarray,
) -> dict[str, float]:
    """MCC, F1 и операционный DL для одного порога score.

    Args:
        tau: Кандидат порога (для записи в результат).
        predicted_anomaly: Маска предсказанных anomaly.
        frame: Ряд с time, lat, lon, x, n, y.
        ground_truth_anomaly: Эталонная маска anomaly.

    Returns:
        dict[str, float]: tau, mcc, f1, precision, recall, loss_m.
    """
    time = frame["time"].to_numpy()
    lon = frame["lon"].to_numpy(dtype=float)
    lat = frame["lat"].to_numpy(dtype=float)
    east_m = frame["x"].to_numpy(dtype=float)
    north_m = frame["n"].to_numpy(dtype=float)
    labels_pred = np.where(predicted_anomaly, 0.0, 1.0)
    labels_gt = np.where(ground_truth_anomaly, 0.0, 1.0)
    prediction = TrajectorySeries(time, lon, lat, labels_pred)
    ground_truth = TrajectorySeries(time, lon, lat, labels_gt)
    precision, recall, f1, mcc = ClassificationMetrics.point_scores(prediction, ground_truth)
    loss_m = operational_distance_loss_m(east_m, north_m, ground_truth_anomaly, predicted_anomaly)
    return {
        "tau": float(tau),
        "mcc": float(mcc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "loss_m": float(loss_m),
    }


def _pick_mcc_band_record(band: list[dict[str, float]], tie_key: str) -> dict[str, float]:
    """Среди записей в полосе MCC — минимальный операционный DL.

    Args:
        band: Записи-кандидаты в полосе MCC.
        tie_key: Имя ключа для детерминированного разрешения ничьей.

    Returns:
        dict[str, float]: Запись с минимальным loss_m, затем с максимальными MCC и F1.
    """
    return sorted(band, key=lambda r: (r["loss_m"], -r["mcc"], -r["f1"], r[tie_key]))[0]


def select_anomaly_threshold(
    frame: pd.DataFrame,
    scores: np.ndarray,
    mcc_band: float,
    n_quantiles: int = 180,
    extra_thresholds: tuple[float, ...] | None = None,
) -> dict[str, float]:
    """Подбирает порог score на train: max MCC, в полосе mcc_band — min операционный DL.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        scores: Anomaly score (больше — подозрительнее).
        mcc_band: Допуск по MCC относительно лучшего значения.
        n_quantiles: Число квантилей в сетке порогов.
        extra_thresholds: Дополнительные кандидаты τ (например, χ²-критическое значение).

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.

    Raises:
        RuntimeError: Нет допустимых порогов.
    """
    scores = np.nan_to_num(np.asarray(scores, dtype=float), nan=0.0, posinf=1e12, neginf=0.0)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    quantiles = np.quantile(scores, np.linspace(0.01, 0.995, n_quantiles))
    if extra_thresholds:
        extras = np.asarray(extra_thresholds, dtype=float)
        candidates = np.unique(np.concatenate([quantiles, extras]))
    else:
        candidates = np.unique(quantiles)
    records = [_threshold_record(float(tau), scores > tau, frame, ground_truth_anomaly) for tau in candidates]
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора порога.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "tau")
    return {
        "tau": chosen["tau"],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
    }


# Порядок и подписи.
TEST_REPORT_METRICS: tuple[tuple[str, str], ...] = (
    ("rmse_m", "RMSE, м"),
    ("max_deviation_m", "Max(d_i), м"),
    ("distance_loss_m", "Distance loss, м"),
    ("delay_median", "Delay median, с"),
    ("delay_mean", "Delay mean, с"),
    ("delay_p95", "Delay p95, с"),
    ("delay_max", "Delay max, с"),
    ("delay_n_on_time", "On-time episodes"),
    ("delay_n_late_or_missed", "Late/missed episodes"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1"),
)


def test_report_row(scenario_name: str, metrics: dict[str, float]) -> dict[str, object]:
    """Оставляет только метрики test в фиксированном порядке.

    Args:
        scenario_name: Имя сценария (df1 / df2).
        metrics: Результат evaluate_test / TrajectoryMetrics.compute_all.

    Returns:
        dict[str, object]: Строка сводной таблицы.
    """
    row: dict[str, object] = {"Сценарий": scenario_name}
    for key, _ in TEST_REPORT_METRICS:
        row[key] = metrics.get(key, float("nan"))
    return row


def _format_metric_value(key: str, value) -> str:
    """Форматирует одну метрику test для таблицы отчёта.

    Args:
        key: Имя метрики из TEST_REPORT_METRICS.
        value: Числовое значение или None.

    Returns:
        str: Строка для display (или «—» при пропуске).
    """
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "—"
    if key in {"precision", "recall", "f1"}:
        rate = float(value)
        return fmt_num(rate, 4) if abs(rate) >= 1000 else f"{rate:.4f}"
    if key in {"distance_loss_m", "rmse_m", "max_deviation_m"}:
        return fmt_num(float(value), 2)
    if key.startswith("delay_n_"):
        return fmt_num(int(value), 0)
    if key.startswith("delay_"):
        delay = float(value)
        return fmt_num(delay, 2) if abs(delay) >= 1000 else f"{delay:.2f}"
    return str(value)


def display_test_report_table(frame: pd.DataFrame, caption: str = "") -> None:
    """Стилизованная таблица итоговых метрик test (как в Отчёте).

    Args:
        frame: Столбец «Сценарий» и ключи из TEST_REPORT_METRICS.
        caption: Подпись таблицы.
    """
    labels = {key: label for key, label in TEST_REPORT_METRICS}
    renamed = frame.rename(columns=labels)
    for key, label in TEST_REPORT_METRICS:
        if label in renamed.columns:
            renamed[label] = renamed[label].map(lambda v, k=key: _format_metric_value(k, v))
    styled = renamed.style
    if caption:
        styled = styled.set_caption(caption)
    display(styled)

## MPCT (Maximum Physically Consistent Trajectory)

### Постановка

Метод строит маску физически допустимого продолжения траектории: последняя принятая точка задаёт якорь для оценки скорости и геометрии пути. Точка, несовместимая с ограничениями по скорости или форме окна, объявляется аномалией; якорь при этом не сдвигается. Параметры $v_+$ (предел скорости от якоря) и $\tau_{\mathrm{shape}}$ (предел индекса формы) не оцениваются по отдельной нормальной выборке — подбираются по сетке на обучающем ряде.

### Score и решение

Пусть $j$ — индекс последней точки с $\hat a_j=0$. Для $i>j$ определяются геодезическая скорость от якоря
$v_{ij}=d_{\mathrm{geo}}(p_j,p_i)/\max(t_i-t_j,1)$ и индекс формы $\mathrm{shape}_i$ по окну длины MPCT_SHAPE_WINDOW точек (отношение длины пути к смещению).

$$\hat a_i=\mathbf{1}\bigl[v_{ij}>v_+\ \lor\ \mathrm{shape}_i>\tau_{\mathrm{shape}}\bigr].$$

На отклонённых точках задаётся непрерывный score, сочетающий превышения $v_+$, $\tau_{\mathrm{shape}}$ и смещение от якоря (км); для принятых точек score равен нулю.

### Обучение и порог

По сеткам MPCT_V_MAX_GRID и MPCT_SHAPE_LIMIT_GRID на train для каждой пары $(v_+,\tau_{\mathrm{shape}})$ строится маска; выбирается пара с максимальным коэффициентом корреляции Мэттьюса, при равенстве в полосе MCC_BAND — с минимальным операционным distance loss. На test применяются те же $v_+$ и $\tau_{\mathrm{shape}}$ без подгонки. Сценарии df1 и df2 — доли обучения TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
MPCT_V_MAX_GRID = (8.0, 10.0, 12.0, 15.0, 20.0)
MPCT_SHAPE_LIMIT_GRID = (1.5, 2.0, 3.0)
MPCT_SHAPE_WINDOW = 6

def mpct_detect(
    frame: pd.DataFrame,
    v_max_mps: float,
    shape_limit: float,
    shape_window: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Жадный якорь: аномалия при превышении скорости или формы окна.

    Args:
        frame: DataFrame с колонками x, n и time.
        v_max_mps: Порог скорости от якоря, м/с.
        shape_limit: Порог track_shape_index.
        shape_window: Окно для track_shape_index.

    Returns:
        tuple[np.ndarray, np.ndarray]: (predicted, scores) — маска anomaly и score.
    """
    east_m = frame["x"].to_numpy(dtype=float)
    north_m = frame["n"].to_numpy(dtype=float)
    time_sec = time_seconds_from_frame(frame)
    step_dist = np.zeros(len(east_m), dtype=float)
    if len(east_m) > 1:
        step_dist[1:] = np.hypot(np.diff(east_m), np.diff(north_m))
    shape = track_shape_index(east_m, north_m, step_dist, shape_window)
    count = len(east_m)
    predicted = np.zeros(count, dtype=bool)
    scores = np.zeros(count, dtype=float)
    last = 0
    for index in range(1, count):
        distance = float(np.hypot(east_m[index] - east_m[last], north_m[index] - north_m[last]))
        delta = max(float(time_sec[index] - time_sec[last]), 1.0)
        speed = distance / delta
        shape_excess = max(0.0, float(shape[index]) - shape_limit)
        if speed > v_max_mps or shape[index] > shape_limit:
            predicted[index] = True
            scores[index] = max(0.0, speed - v_max_mps) + shape_excess + distance / 1000.0
        else:
            last = index
    return predicted, scores

def select_mpct_hyperparameters(
    train: pd.DataFrame,
    v_max_grid: tuple[float, ...],
    shape_limit_grid: tuple[float, ...],
    shape_window: int,
    mcc_band: float,
) -> dict[str, object]:
    """Сетка $v_+$ и $\tau_{\mathrm{shape}}$: max MCC, полоса mcc_band — min операционный DL.

    Args:
        train: Обучающий DataFrame.
        v_max_grid: Кандидаты порога скорости, м/с.
        shape_limit_grid: Кандидаты порога формы окна.
        shape_window: Окно для track_shape_index.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, object]: v_max_mps, shape_limit, train_mcc, train_f1, train_dl_m,
        best_mcc, band_size.
    """
    ground_truth_anomaly = train["y"].astype(str).to_numpy() == "anomaly"
    records: list[dict[str, float]] = []
    for v_max_mps in v_max_grid:
        for shape_limit in shape_limit_grid:
            predicted, _ = mpct_detect(
                train, float(v_max_mps), float(shape_limit), shape_window
            )
            record = _threshold_record(float(v_max_mps), predicted, train, ground_truth_anomaly)
            record["v_max_mps"] = float(v_max_mps)
            record["shape_limit"] = float(shape_limit)
            print(
                f"  v_max {v_max_mps:.0f} м/с, форма {shape_limit:.1f}: "
                f"MCC {record['mcc']:.3f}, DL {record['loss_m'] / 1000:.4f} км, "
                f"F1 {record['f1']:.3f}"
            )
            records.append(record)
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора MPCT.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "v_max_mps")
    return {
        "v_max_mps": chosen["v_max_mps"],
        "shape_limit": chosen["shape_limit"],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
    }

mpct_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

mpct_rows: list[dict[str, object]] = []
mpct_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in mpct_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"MPCT: {scenario_name}")
    started = time.perf_counter()
    print("  подбор v_max и предела формы на train...")
    selection = select_mpct_hyperparameters(
        train,
        MPCT_V_MAX_GRID,
        MPCT_SHAPE_LIMIT_GRID,
        MPCT_SHAPE_WINDOW,
        MCC_BAND,
    )
    v_max = float(selection["v_max_mps"])
    shape_limit = float(selection["shape_limit"])
    print(
        f"  выбрано v_max {v_max:.0f} м/с, форма ≤ {shape_limit:.1f}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    predicted_test, test_scores = mpct_detect(test, v_max, shape_limit, MPCT_SHAPE_WINDOW)
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    mpct_rows.append(test_report_row(scenario_name, metrics))
    mpct_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
            "test_scores": test_scores,
        }
    )

mpct_summary = pd.DataFrame(mpct_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    mpct_summary.loc[:, report_columns],
    caption="MPCT: сводные метрики на test",
)

_mpct_ideal_test = mpct_scenario_results[0]["test"]
_mpct_map_tail = min(PLOT_TAIL, len(_mpct_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_mpct_map_tail)} точек теста df1 (bbox: move + 5%)")
_mpct_dummy_move = np.zeros(len(_mpct_ideal_test), dtype=bool)
_mpct_truth_pred, _mpct_truth_gt = trajectory_series_pair(_mpct_ideal_test, _mpct_dummy_move)
_mpct_truth_pred = slice_trajectory_series(_mpct_truth_pred, _mpct_map_tail)
_mpct_truth_gt = slice_trajectory_series(_mpct_truth_gt, _mpct_map_tail)
plot_track_maps(
    _mpct_truth_pred,
    _mpct_truth_gt,
    map_panel="ground_truth",
    title="MPCT — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in mpct_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"MPCT — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

## AR($p$) для детекции аномалий траектории

### Постановка

Авторегрессия порядка $p$ задаёт линейную зависимость текущего уровня ряда от $p$ предыдущих значений с дрейфом и гауссовским шумом. Детектор опирается на причинный одношаговый прогноз: большая ошибка прогноза трактуется как признак аномалии.

### Модель и score

Для координат ENU (метры) используются две независимые модели AR($p$) с линейным трендом на уровнях:

$$
x_i = c_x + \sum_{k=1}^{p}\phi_{x,k}\, x_{i-k} + \varepsilon_i, \qquad
n_i = c_n + \sum_{k=1}^{p}\phi_{n,k}\, n_{i-k} + \eta_i.
$$

Параметры оцениваются по всему обучающему ряду (включая участки с $y_i=\mathrm{anomaly}$). Одношаговые прогнозы $\hat x_i$, $\hat n_i$ строятся причинно. Скалярный score и решение:

$$
s_i = \sqrt{(x_i-\hat x_i)^2+(n_i-\hat n_i)^2}, \qquad \hat a_i=\mathbf{1}[s_i>\tau].
$$

В эксперименте $p=$ AR_LAGS (по умолчанию 2).

### Обучение и порог

Порог $\tau$ подбирается только на train по сетке квантилей $\{s_i\}$: максимум коэффициента корреляции Мэттьюса; в полосе MCC_BAND — минимальный операционный distance loss. На test — тот же $\tau$ и те же оценённые коэффициенты AR. Сценарии df1 и df2 соответствуют TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
AR_LAGS = 2

def fit_ar_series(series: np.ndarray, lags: int):
    """Оценивает AR(p) на уровнях ряда (ARIMA(p,0,0), trend='t').

    Args:
        series: Одномерная координата ENU, м.
        lags: Порядок p.

    Returns:
        Результат .fit() statsmodels.
    """
    order = (lags, 0, 0)
    model = ARIMA(
        np.asarray(series, dtype=np.float64),
        order=order,
        trend="t",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit()


def ar_one_step_levels(series: np.ndarray, fit_result, lags: int) -> np.ndarray:
    """Причинные одношаговые прогнозы уровня для AR(p).

    Args:
        series: Координата ENU.
        fit_result: Параметры, оцененные на train.
        lags: Порядок p.

    Returns:
        np.ndarray: Прогноз уровня той же длины, что series.
    """
    series = np.asarray(series, dtype=np.float64)
    order = (lags, 0, 0)
    model = ARIMA(
        series,
        order=order,
        trend="t",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    filtered = model.filter(fit_result.params)
    prediction = filtered.get_prediction(start=0, end=len(series) - 1, dynamic=True)
    predicted = np.asarray(prediction.predicted_mean, dtype=np.float64)
    predicted[0] = series[0]
    return np.nan_to_num(predicted, nan=series[0])


def ar_enu_scores(
    east_m: np.ndarray,
    north_m: np.ndarray,
    fit_east,
    fit_north,
    lags: int,
) -> np.ndarray:
    """Score в метрах ENU: гипотенуза ошибок одношагового прогноза AR.

    Args:
        east_m: Восток.
        north_m: Север.
        fit_east: AR по x с train.
        fit_north: AR по n с train.
        lags: Порядок p.

    Returns:
        np.ndarray: Скалярный score на каждой точке.
    """
    east_hat = ar_one_step_levels(east_m, fit_east, lags)
    north_hat = ar_one_step_levels(north_m, fit_north, lags)
    return np.hypot(np.asarray(east_m, dtype=float) - east_hat, np.asarray(north_m, dtype=float) - north_hat)


ar_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

ar_rows: list[dict[str, object]] = []
ar_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in ar_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"AR({AR_LAGS}): {scenario_name}")
    started = time.perf_counter()
    fit_east = fit_ar_series(train["x"].to_numpy(dtype=float), AR_LAGS)
    fit_north = fit_ar_series(train["n"].to_numpy(dtype=float), AR_LAGS)
    train_scores = ar_enu_scores(
        train["x"].to_numpy(dtype=float),
        train["n"].to_numpy(dtype=float),
        fit_east,
        fit_north,
        AR_LAGS,
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  порог {selection['tau']:.4g} м: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = ar_enu_scores(
        test["x"].to_numpy(dtype=float),
        test["n"].to_numpy(dtype=float),
        fit_east,
        fit_north,
        AR_LAGS,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    ar_rows.append(test_report_row(scenario_name, metrics))
    ar_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

ar_summary = pd.DataFrame(ar_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    ar_summary.loc[:, report_columns],
    caption=f"AR({AR_LAGS}): сводные метрики на test",
)

_ar_ideal_test = ar_scenario_results[0]["test"]
_ar_map_tail = min(PLOT_TAIL, len(_ar_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_ar_map_tail)} точек теста df1 (bbox: move + 5%)")
_ar_dummy_move = np.zeros(len(_ar_ideal_test), dtype=bool)
_ar_truth_pred, _ar_truth_gt = trajectory_series_pair(_ar_ideal_test, _ar_dummy_move)
_ar_truth_pred = slice_trajectory_series(_ar_truth_pred, _ar_map_tail)
_ar_truth_gt = slice_trajectory_series(_ar_truth_gt, _ar_map_tail)
plot_track_maps(
    _ar_truth_pred,
    _ar_truth_gt,
    map_panel="ground_truth",
    title=f"AR({AR_LAGS}) — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in ar_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"AR({AR_LAGS}) — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## ARIMA для детекции аномалий траектории

### Постановка

Модель ARIMA$(p,d,q)$ описывает стохастический ряд как сочетание авторегрессии, интегрирования и скользящего среднего. В детекторе на каждом шаге строится причинный одношаговый прогноз уровня; аномалия — большое расхождение наблюдения с прогнозом.

### Модель и score

Для одномерного ряда $x_t$:

$$
\phi(B)(1-B)^d x_t = c + \theta(B)\varepsilon_t, \qquad \varepsilon_t \sim \mathcal N(0,\sigma^2),
$$

где $B$ — оператор сдвига, $\phi(B)=1-\phi_1 B-\cdots-\phi_p B^p$, $\theta(B)=1+\theta_1 B+\cdots+\theta_q B^q$.

В эксперименте фиксирован порядок $(p,d,q)=(2,1,1)$. Для координат ENU (восток $x_i$, север $n_i$, м) задаются две независимые модели $\mathrm{ARIMA}(2,1,1)$; параметры оцениваются по всему train. Скалярный score:

$$
s_i = \sqrt{(x_i-\hat{x}_i)^2 + (n_i-\hat{n}_i)^2}, \qquad \hat a_i = \mathbf{1}[s_i > \tau].
$$

Положительный класс для метрик классификации — $\mathrm{anomaly}$.

### Обучение и порог

Порог $\tau$ выбирается только на train по квантилям $\{s_i\}$: максимум MCC; в полосе MCC_BAND — минимальный операционный distance loss. На test применяется тот же $\tau$ без подгонки. Сценарии df1 и df2 — доли обучения TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
def arima_trend_for_order(order: tuple[int, int, int]) -> str:
    """trend для statsmodels: при d>0 нужен 't', не 'c'.

    Args:
        order: (p, d, q).

    Returns:
        str: 't' или 'c'.
    """
    return "t" if order[1] > 0 else "c"


def fit_arima_series(series: np.ndarray, order: tuple[int, int, int]):
    """Оценивает ARIMA на всём переданном ряду.

    Args:
        series: Одномерный ряд координаты.
        order: Порядок (p, d, q).

    Returns:
        Результат .fit() statsmodels.
    """
    model = ARIMA(
        np.asarray(series, dtype=np.float64),
        order=order,
        trend=arima_trend_for_order(order),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit()


def arima_one_step_levels(series: np.ndarray, fit_result, order: tuple[int, int, int]) -> np.ndarray:
    """Причинные одношаговые прогнозы уровня ряда.

    Args:
        series: Координата ENU.
        fit_result: Обученная модель (параметры с train).
        order: Порядок ARIMA.

    Returns:
        np.ndarray: Прогноз уровня той же длины, что series.
    """
    series = np.asarray(series, dtype=np.float64)
    model = ARIMA(
        series,
        order=order,
        trend=arima_trend_for_order(order),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    filtered = model.filter(fit_result.params)
    prediction = filtered.get_prediction(start=0, end=len(series) - 1, dynamic=True)
    predicted = np.asarray(prediction.predicted_mean, dtype=np.float64)
    predicted[0] = series[0]
    return np.nan_to_num(predicted, nan=series[0])


def arima_enu_scores(
    east_m: np.ndarray,
    north_m: np.ndarray,
    fit_east,
    fit_north,
    order: tuple[int, int, int],
) -> np.ndarray:
    """Score в метрах ENU: гипотенуза ошибок прогноза x и n.

    Args:
        east_m: Восток.
        north_m: Север.
        fit_east: Модель по x с train.
        fit_north: Модель по n с train.
        order: Порядок ARIMA.

    Returns:
        np.ndarray: Скалярный score на каждой точке.
    """
    east_hat = arima_one_step_levels(east_m, fit_east, order)
    north_hat = arima_one_step_levels(north_m, fit_north, order)
    return np.hypot(np.asarray(east_m, dtype=float) - east_hat, np.asarray(north_m, dtype=float) - north_hat)



ARIMA_ORDER = (2, 1, 1)

SCENARIOS = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

arima_rows: list[dict[str, object]] = []
scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in SCENARIOS:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(scenario_name)
    started = time.perf_counter()
    fit_east = fit_arima_series(train["x"].to_numpy(dtype=float), ARIMA_ORDER)
    fit_north = fit_arima_series(train["n"].to_numpy(dtype=float), ARIMA_ORDER)
    train_scores = arima_enu_scores(
        train["x"].to_numpy(dtype=float),
        train["n"].to_numpy(dtype=float),
        fit_east,
        fit_north,
        ARIMA_ORDER,
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  порог {selection['tau']:.4g} м: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = arima_enu_scores(
        test["x"].to_numpy(dtype=float),
        test["n"].to_numpy(dtype=float),
        fit_east,
        fit_north,
        ARIMA_ORDER,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started

    arima_rows.append(test_report_row(scenario_name, metrics))
    scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

summary = pd.DataFrame(arima_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    summary.loc[:, report_columns],
    caption="ARIMA(2,1,1): сводные метрики на test",
)

_ideal_test = scenario_results[0]["test"]
_map_tail = min(PLOT_TAIL, len(_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_map_tail)} точек теста df1 (bbox: move + 5%)")
_dummy_move = np.zeros(len(_ideal_test), dtype=bool)
_truth_pred, _truth_gt = trajectory_series_pair(_ideal_test, _dummy_move)
_truth_pred = slice_trajectory_series(_truth_pred, _map_tail)
_truth_gt = slice_trajectory_series(_truth_gt, _map_tail)
plot_track_maps(
    _truth_pred,
    _truth_gt,
    map_panel="ground_truth",
    title="ARIMA — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"ARIMA — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Фильтр Калмана (постоянная скорость)

### Постановка

Скрытое состояние — положение и скорость в плоскости ENU; измерение — координаты $(x_i, n_i)$. Аномалия — точка, слабо согласованная с причинным предсказанием фильтра до учёта текущего измерения.

### Модель и score

$$
\mathbf{x}_i = (p_x, p_y, v_x, v_y)^\top, \quad \mathbf{x}_i = A(\Delta t_i)\mathbf{x}_{i-1} + \mathbf{w}_i, \quad \mathbf{z}_i = H\mathbf{x}_i + \mathbf{v}_i, \quad H=\begin{pmatrix}1&0&0&0\\0&1&0&0\end{pmatrix}.
$$

Шум процесса моделируется дискретным белым ускорением с $\sigma_{\mathrm{acc}}$; ковариация измерения — $\sigma_{\mathrm{meas}}^2 I_2$. Интервал $\Delta t_i$ ограничивается сверху; при разрыве по времени выше KALMAN_GAP_SEC состояние переинициализируется по измерению (скорости обнуляются), нормированная инновация текущего шага сохраняется в score.

Инновация $\mathbf{r}_i = \mathbf{z}_i - H\hat{\mathbf{x}}_{i|i-1}$, ковариация $S_i$. Anomaly score: $s_i = \mathbf{r}_i^\top S_i^{-1}\mathbf{r}_i.$

При согласованной гауссовской модели $s_i$ ориентируется на $\chi^2_2$. Решение: $\hat a_i = \mathbf{1}[s_i > \tau]$.

### Обучение и порог

Порог $\tau$ подбирается на train по квантилям $\{s_i\}$ и ориентиру $\chi^2_{0{,}999}(2)$: максимум MCC; в полосе MCC_BAND — минимальный операционный distance loss. $\sigma_{\mathrm{acc}}$, $\sigma_{\mathrm{meas}}$ и параметры разрывов (KALMAN_GAP_SEC, KALMAN_DT_CAP_SEC) фиксированы. На test — тот же $\tau$, фильтр прогоняется по тестовому ряду. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
KALMAN_SIGMA_ACC = 0.35
KALMAN_SIGMA_MEAS = 40.0
KALMAN_GAP_SEC = 180.0
KALMAN_DT_CAP_SEC = 45.0

def kalman_process_noise(dt: float, sigma_acc: float) -> np.ndarray:
    """Ковариация шума процесса CV при дискретном белом ускорении.

    Args:
        dt: Шаг модели, с.
        sigma_acc: СКО ускорения, м/с².

    Returns:
        np.ndarray: Матрица 4×4.
    """
    dt2 = dt * dt
    dt3 = dt2 * dt
    dt4 = dt2 * dt2
    noise = np.zeros((4, 4))
    noise[0, 0] = noise[1, 1] = dt4 / 4.0
    noise[2, 2] = noise[3, 3] = dt2
    noise[0, 2] = noise[2, 0] = noise[1, 3] = noise[3, 1] = dt3 / 2.0
    return noise * (sigma_acc**2)


def kalman_nis_scores(
    east_m: np.ndarray,
    north_m: np.ndarray,
    time_sec: np.ndarray,
    sigma_acc: float,
    sigma_meas: float,
    gap_sec: float,
    dt_cap_sec: float,
) -> np.ndarray:
    """Нормированный квадрат инновации на каждой точке ENU.

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        time_sec: Время, с.
        sigma_acc: СКО ускорения процесса.
        sigma_meas: СКО ошибки измерения координат.
        gap_sec: Порог разрыва для переинициализации.
        dt_cap_sec: Верхняя граница Δt в модели.

    Returns:
        np.ndarray: Скалярный score (больше — подозрительнее).
    """
    xy = np.column_stack(
        [np.asarray(east_m, dtype=np.float64), np.asarray(north_m, dtype=np.float64)]
    )
    t_sec = np.asarray(time_sec, dtype=np.float64)
    count = len(xy)
    state = np.zeros(4)
    state[:2] = xy[0]
    cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    observation = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
    meas_cov = np.eye(2) * (sigma_meas**2)
    eye = np.eye(4)
    scores = np.zeros(count)
    for index in range(1, count):
        dt = float(t_sec[index] - t_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > gap_sec
        dt_model = min(dt, dt_cap_sec)
        transition = np.eye(4)
        transition[0, 2] = dt_model
        transition[1, 3] = dt_model
        pred = transition @ state
        pred_cov = transition @ cov @ transition.T + kalman_process_noise(dt_model, sigma_acc)
        innov = xy[index] - observation @ pred
        innov_cov = observation @ pred_cov @ observation.T + meas_cov
        solved = np.linalg.solve(innov_cov, innov)
        scores[index] = float(innov @ solved)
        gain = pred_cov @ observation.T @ np.linalg.inv(innov_cov)
        state = pred + gain @ innov
        cov = (eye - gain @ observation) @ pred_cov
        cov = 0.5 * (cov + cov.T)
        if gap:
            state = np.array([xy[index, 0], xy[index, 1], 0.0, 0.0])
            cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    return scores


KALMAN_CHI999 = float(chi2.ppf(0.999, 2))

kalman_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

kalman_rows: list[dict[str, object]] = []
kalman_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in kalman_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Kalman CV: {scenario_name}")
    started = time.perf_counter()
    train_scores = kalman_nis_scores(
        train["x"].to_numpy(dtype=float),
        train["n"].to_numpy(dtype=float),
        time_seconds_from_frame(train),
        KALMAN_SIGMA_ACC,
        KALMAN_SIGMA_MEAS,
        KALMAN_GAP_SEC,
        KALMAN_DT_CAP_SEC,
    )
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=(KALMAN_CHI999,),
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана NIS на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2) = {KALMAN_CHI999:.2f}; порог NIS {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = kalman_nis_scores(
        test["x"].to_numpy(dtype=float),
        test["n"].to_numpy(dtype=float),
        time_seconds_from_frame(test),
        KALMAN_SIGMA_ACC,
        KALMAN_SIGMA_MEAS,
        KALMAN_GAP_SEC,
        KALMAN_DT_CAP_SEC,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    kalman_rows.append(test_report_row(scenario_name, metrics))
    kalman_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

kalman_summary = pd.DataFrame(kalman_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    kalman_summary.loc[:, report_columns],
    caption="Kalman CV: сводные метрики на test",
)

_kalman_ideal_test = kalman_scenario_results[0]["test"]
_kalman_map_tail = min(PLOT_TAIL, len(_kalman_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_kalman_map_tail)} точек теста df1 (bbox: move + 5%)")
_kalman_dummy_move = np.zeros(len(_kalman_ideal_test), dtype=bool)
_kalman_truth_pred, _kalman_truth_gt = trajectory_series_pair(_kalman_ideal_test, _kalman_dummy_move)
_kalman_truth_pred = slice_trajectory_series(_kalman_truth_pred, _kalman_map_tail)
_kalman_truth_gt = slice_trajectory_series(_kalman_truth_gt, _kalman_map_tail)
plot_track_maps(
    _kalman_truth_pred,
    _kalman_truth_gt,
    map_panel="ground_truth",
    title="Kalman CV — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in kalman_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"Kalman CV — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Gaussian process

### Постановка

Приращения координат $(\Delta x_i, \Delta n_i)$ во времени моделируются двумя независимыми гауссовскими процессами регрессии (восток и север). Точка подозрительна, если фактический шаг существенно отклоняется от GP-прогноза с учётом неопределённости.

### Score

Ядро RBF по времени $t$; для шага $i$ прогноз $f(t_i)\mid\mathcal{D}\sim\mathcal{N}(\mu_i,\sigma_i^2)$ по опорным моментам на train: $s_i = \frac{(\Delta x_i-\mu_x(t_i))^2}{\sigma_x^2(t_i)} + \frac{(\Delta n_i-\mu_n(t_i))^2}{\sigma_n^2(t_i)}.$

При независимых гауссовских остатках $s_i$ ориентируется на $\chi^2_2$. Решение $\hat a_i=\mathbf{1}[s_i>\tau]$.

### Обучение и порог

Опорные моменты — равномерная подвыборка точек $y_i=\mathrm{move}$ на train (число GP_SUPPORT_COUNT). Длина корреляции и шум наблюдения задаются константами GP_LENGTHSCALE_HOURS, GP_NOISE_MIN_M, GP_NOISE_STEP_FRAC; подгонка гиперпараметров по marginal likelihood не выполняется. Порог $\tau$ на train: квантили $s_i$, кандидат $\chi^2_{0{,}999}(2)$, максимум MCC, в полосе MCC_BAND — минимальный операционный distance loss. На test — те же опоры, гиперпараметры и $\tau$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
GP_SUPPORT_COUNT = 250
GP_LENGTHSCALE_HOURS = 6.0
GP_NOISE_MIN_M = 5.0
GP_NOISE_STEP_FRAC = 0.25

def gp_move_support_indices(frame: pd.DataFrame, support_count: int) -> np.ndarray:
    """Равномерные индексы move-точек train для опор GP.

    Args:
        frame: Обучающий DataFrame с колонкой y.
        support_count: Желаемое число опорных точек.

    Returns:
        np.ndarray: Индексы опорных move-точек train.
    """
    move_index = np.flatnonzero(frame["y"].astype(str).to_numpy() == "move")
    if move_index.size == 0:
        raise RuntimeError("На train нет точек move для опор GP.")
    count = min(int(support_count), move_index.size)
    grid = np.linspace(0, move_index.size - 1, count).astype(int)
    return move_index[grid]

def gp_step_hyperparameters(dx_support: np.ndarray, dn_support: np.ndarray) -> tuple[float, float]:
    """Фиксированные ℓ (ч) и шум шага, м (как example_model_1).

    Args:
        dx_support: Шаги востока на опорах, м.
        dn_support: Шаги севера на опорах, м.

    Returns:
        tuple[float, float]: (lengthscale_sec, noise_m).
    """
    lengthscale_sec = float(GP_LENGTHSCALE_HOURS * 3600.0)
    step_scale = float(np.median(np.hypot(dx_support, dn_support)))
    noise_m = float(max(GP_NOISE_MIN_M, GP_NOISE_STEP_FRAC * step_scale))
    return lengthscale_sec, noise_m



def _fit_fixed_gpr(
    time_support: np.ndarray,
    step_support: np.ndarray,
    lengthscale_sec: float,
    noise_m: float,
) -> tuple[GaussianProcessRegressor, float]:
    """Один GPR по времени для шага dx или dn (optimizer=None).

    Args:
        time_support: Время опорных точек, с.
        step_support: Шаг ENU на опорах, м.
        lengthscale_sec: Длина корреляции RBF, с.
        noise_m: Шум шага (WhiteKernel), м.

    Returns:
        tuple[GaussianProcessRegressor, float]: обученный GPR и среднее шага (center).
    """
    time_col = np.asarray(time_support, dtype=np.float64).reshape(-1, 1)
    steps = np.asarray(step_support, dtype=np.float64)
    center = float(np.mean(steps))
    signal = float(np.std(steps) + 1.0)
    kernel = (
        ConstantKernel(constant_value=signal**2, constant_value_bounds="fixed")
        * RBF(length_scale=lengthscale_sec, length_scale_bounds="fixed")
        + WhiteKernel(noise_level=noise_m**2, noise_level_bounds="fixed")
    )
    regressor = GaussianProcessRegressor(kernel=kernel, optimizer=None, normalize_y=False)
    regressor.fit(time_col, steps - center)
    return regressor, center


def _maritime_gp_scores(
    frame: pd.DataFrame,
    gpr_x: GaussianProcessRegressor,
    center_x: float,
    gpr_n: GaussianProcessRegressor,
    center_n: float,
) -> np.ndarray:
    """χ²-тип score по уже обученным GPR востока и севера.

    Args:
        frame: DataFrame с колонками x, n и time.
        gpr_x: GPR шага востока.
        center_x: Среднее шага востока на опорах, м.
        gpr_n: GPR шага севера.
        center_n: Среднее шага севера на опорах, м.

    Returns:
        np.ndarray: Score длины N (сумма двух нормированных квадратов).
    """
    dx, dn = enu_step_components(frame["x"].to_numpy(dtype=float), frame["n"].to_numpy(dtype=float))
    time_query = time_seconds_from_frame(frame).reshape(-1, 1)
    mean_x, std_x = gpr_x.predict(time_query, return_std=True)
    mean_n, std_n = gpr_n.predict(time_query, return_std=True)
    var_x = np.clip(std_x**2, 1e-6, None)
    var_n = np.clip(std_n**2, 1e-6, None)
    return (dx - mean_x - center_x) ** 2 / var_x + (dn - mean_n - center_n) ** 2 / var_n


GP_CHI999 = float(chi2.ppf(0.999, 2))

gp_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

gp_rows: list[dict[str, object]] = []
gp_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in gp_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Gaussian process (sklearn): {scenario_name}")
    started = time.perf_counter()
    support_idx = gp_move_support_indices(train, GP_SUPPORT_COUNT)
    train_time = time_seconds_from_frame(train)
    train_dx, train_dn = enu_step_components(
        train["x"].to_numpy(dtype=float),
        train["n"].to_numpy(dtype=float),
    )
    time_support = train_time[support_idx]
    dx_support = train_dx[support_idx]
    dn_support = train_dn[support_idx]
    lengthscale_sec, noise_m = gp_step_hyperparameters(dx_support, dn_support)
    gpr_x, center_x = _fit_fixed_gpr(time_support, dx_support, lengthscale_sec, noise_m)
    gpr_n, center_n = _fit_fixed_gpr(time_support, dn_support, lengthscale_sec, noise_m)

    train_scores = _maritime_gp_scores(train, gpr_x, center_x, gpr_n, center_n)
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=(GP_CHI999,),
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  опор {len(support_idx)}, ℓ {lengthscale_sec / 3600:.1f} ч, шум шага {noise_m:.1f} м; "
        f"χ²₀.₉₉₉(df=2)={GP_CHI999:.2f}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = _maritime_gp_scores(test, gpr_x, center_x, gpr_n, center_n)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    gp_rows.append(test_report_row(scenario_name, metrics))
    gp_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

gp_summary = pd.DataFrame(gp_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    gp_summary.loc[:, report_columns],
    caption="GP (sklearn): сводные метрики на test",
)

_gp_ideal_test = gp_scenario_results[0]["test"]
_gp_map_tail = min(PLOT_TAIL, len(_gp_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_gp_map_tail)} точек теста df1 (bbox: move + 5%)")
_gp_dummy_move = np.zeros(len(_gp_ideal_test), dtype=bool)
_gp_truth_pred, _gp_truth_gt = trajectory_series_pair(_gp_ideal_test, _gp_dummy_move)
_gp_truth_pred = slice_trajectory_series(_gp_truth_pred, _gp_map_tail)
_gp_truth_gt = slice_trajectory_series(_gp_truth_gt, _gp_map_tail)
plot_track_maps(
    _gp_truth_pred,
    _gp_truth_gt,
    map_panel="ground_truth",
    title="GP — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in gp_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"GP — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## GP-EKF для детекции аномалий траектории

### Постановка

Расширенный фильтр Калмана с моделью постоянной скорости в ENU: детерминированный переход дополняется остатком, обученным гауссовским процессом по признакам перехода $(v^E, v^N, \Delta t)$; измерение линейно по положению $(E,N)$.

### Модель и score

Состояние $\mathbf{x}_i=(E,N,v^E,v^N)^\top$; базовый сдвиг — модель CV, GP задаёт остаток перехода; якобиан среднего GP входит в линеаризацию EKF.

Anomaly score — нормированная квадратичная инновация измерения: $s_i=\mathbf{r}_i^\top S_i^{-1}\mathbf{r}_i, \qquad \hat a_i=\mathbf{1}[s_i>\tau].$

### Обучение и порог

Опорные переходы отбираются среди точек $y_i=\mathrm{move}$ на train (до GP_EKF_SUPPORT). Гиперпараметры RBF (длина корреляции, шум) выбираются по сетке с максимизацией лог-правдоподобия на опорах. Порог $\tau$ на train: квантили $s_i$, $\chi^2_{0{,}999}(2)$, максимум MCC, полоса MCC_BAND, минимальный операционный distance loss. Параметры измерения и разрывов (GP_EKF_SIGMA_MEAS, GP_EKF_MAX_DT_SEC, GP_EKF_GAP_SEC) фиксированы. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
GP_EKF_SUPPORT = 160
GP_EKF_SIGMA_MEAS = 40.0
GP_EKF_MAX_DT_SEC = 45.0
GP_EKF_GAP_SEC = 180.0

def gp_ekf_transition_sample(
    xy: np.ndarray,
    velocity_east: np.ndarray,
    velocity_north: np.ndarray,
    step_dt: np.ndarray,
    move_mask: np.ndarray,
    max_dt: float,
    support: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Опорные штатные переходы: вход (v_E, v_N, Δt), остаток CV.

    Args:
        xy: Координаты ENU shape (N, 2), м.
        velocity_east: Входящая скорость востока, м/с.
        velocity_north: Входящая скорость севера, м/с.
        step_dt: Шаги времени, с.
        move_mask: Маска move-точек.
        max_dt: Максимальный Δt перехода, с.
        support: Максимальное число опорных переходов.

    Returns:
        tuple[np.ndarray, np.ndarray]: inputs (n, 3) и targets (n, 4) — остатки
        положения и скорости.
    """
    valid = np.asarray(move_mask, dtype=bool)
    count = len(xy)
    usable: list[int] = []
    for index in range(1, count - 1):
        dt = float(step_dt[index + 1])
        if dt <= 0.0 or dt > max_dt:
            continue
        if not valid[index] or not valid[index + 1]:
            continue
        usable.append(index)
    if len(usable) < 8:
        raise RuntimeError("Для GP-EKF нужно хотя бы 8 штатных переходов с Δt ≤ max_dt")
    chosen = np.asarray(usable, dtype=int)
    if len(chosen) > support:
        grid = np.linspace(0, len(chosen) - 1, support).astype(int)
        chosen = chosen[grid]
    inputs = np.column_stack(
        [velocity_east[chosen], velocity_north[chosen], step_dt[chosen + 1]]
    )
    dt = step_dt[chosen + 1]
    residual_east = (xy[chosen + 1, 0] - xy[chosen, 0]) - velocity_east[chosen] * dt
    residual_north = (xy[chosen + 1, 1] - xy[chosen, 1]) - velocity_north[chosen] * dt
    residual_ve = velocity_east[chosen + 1] - velocity_east[chosen]
    residual_vn = velocity_north[chosen + 1] - velocity_north[chosen]
    targets = np.column_stack([residual_east, residual_north, residual_ve, residual_vn])
    return inputs, targets



def _gp_ekf_unit_kernel(features: np.ndarray, lengthscale: float) -> np.ndarray:
    """RBF-ядро без обучаемого масштаба сигнала (единичная амплитуда).

    Args:
        features: Нормированные входы shape (n, 3).
        lengthscale: Длина корреляции RBF.

    Returns:
        np.ndarray: Матрица ядра shape (n, n).
    """
    scale = features / lengthscale
    gram = scale @ scale.T
    sq = np.sum(scale * scale, axis=1)
    return np.exp(-0.5 * (sq[:, None] + sq[None, :] - 2.0 * gram))


def _gp_ekf_cholesky(features: np.ndarray, lengthscale: float, noise: float) -> np.ndarray:
    """Разложение Холецкого ковариации GP на опорных переходах.

    Args:
        features: Нормированные входы shape (n, 3).
        lengthscale: Длина корреляции RBF.
        noise: Шум ядра.

    Returns:
        np.ndarray: Нижнетреугольный множитель L, K = L Lᵀ.
    """
    gram = _gp_ekf_unit_kernel(features, lengthscale)
    gram.flat[:: len(features) + 1] += noise**2
    jitter = 1e-8
    for _ in range(6):
        try:
            return np.linalg.cholesky(gram)
        except LinAlgError:
            gram.flat[:: len(features) + 1] += jitter
            jitter *= 10.0
    raise LinAlgError("Ковариация GP-EKF не разложилась")


def _gp_ekf_chol_solve(chol: np.ndarray, vector: np.ndarray) -> np.ndarray:
    """Решение K^{-1} v при K = L L^T.

    Args:
        chol: Множитель Холецкого L.
        vector: Вектор правой части.

    Returns:
        np.ndarray: Вектор K^{-1} v.
    """
    solved = solve_triangular(chol, vector, lower=True, check_finite=False)
    return solve_triangular(chol.T, solved, lower=False, check_finite=False)


def _gp_ekf_select_hyperparameters(
    features: np.ndarray, targets: np.ndarray
) -> tuple[float, float, np.ndarray, np.ndarray]:
    """Сетка (ℓ, σ) по суммарному log ML.

    Args:
        features: Нормированные входы shape (n, 3).
        targets: Нормированные остатки shape (n, 4).

    Returns:
        tuple[float, float, np.ndarray, np.ndarray]: lengthscale, noise, chol, alpha.
    """
    best = None
    for lengthscale in (0.6, 1.0, 1.5, 2.2):
        for noise in (0.15, 0.3, 0.5):
            try:
                chol = _gp_ekf_cholesky(features, lengthscale, noise)
            except LinAlgError:
                continue
            score = 0.0
            alphas = []
            for column in range(targets.shape[1]):
                alpha = _gp_ekf_chol_solve(chol, targets[:, column])
                alphas.append(alpha)
                quad = float(targets[:, column] @ alpha)
                log_det = 2.0 * float(np.sum(np.log(np.diag(chol))))
                score += -0.5 * quad - 0.5 * log_det
            if best is None or score > best[0]:
                best = (score, lengthscale, noise, chol, np.column_stack(alphas))
    if best is None:
        raise LinAlgError("Нет допустимых гиперпараметров GP-EKF")
    _, lengthscale, noise, chol, alpha = best
    return lengthscale, noise, chol, alpha


def fit_gp_ekf_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Обучает GP остатка перехода CV на штатных переходах train.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, object]: Параметры GP, нормировки и опорные величины
        (n_support, lengthscale, noise, train_z, chol, alpha, inv_length_sq, …).
    """
    xy = enu_xy_from_frame(train)
    time_sec = time_seconds_from_frame(train)
    move_mask = train["y"].astype(str).to_numpy() == "move"
    v_e, v_n, step_dt = incoming_velocity_enu(xy[:, 0], xy[:, 1], time_sec)
    inputs, targets = gp_ekf_transition_sample(
        xy, v_e, v_n, step_dt, move_mask, GP_EKF_MAX_DT_SEC, GP_EKF_SUPPORT
    )
    input_center = np.mean(inputs, axis=0)
    input_scale = np.std(inputs, axis=0)
    input_scale = np.where(input_scale < 1e-3, 1.0, input_scale)
    target_center = np.mean(targets, axis=0)
    target_scale = np.std(targets, axis=0)
    target_scale = np.maximum(target_scale, np.array([4.0, 4.0, 0.08, 0.08]))
    train_z = (inputs - input_center) / input_scale
    train_y = (targets - target_center) / target_scale
    lengthscale, noise, chol, alpha = _gp_ekf_select_hyperparameters(train_z, train_y)
    return {
        "n_support": int(len(train_z)),
        "lengthscale": float(lengthscale),
        "noise": float(noise),
        "input_center": input_center,
        "input_scale": input_scale,
        "target_center": target_center,
        "target_scale": target_scale,
        "train_z": train_z,
        "chol": chol,
        "alpha": alpha,
        "inv_length_sq": np.full(3, 1.0 / (lengthscale**2)),
    }


def _gp_ekf_features(model: dict[str, object], velocity_east: float, velocity_north: float, dt: float) -> np.ndarray:
    """Нормированный вход GP перехода (v_E, v_N, Δt).

    Args:
        model: Словарь обученной модели из fit_gp_ekf_on_train.
        velocity_east: Входящая скорость востока, м/с.
        velocity_north: Входящая скорость севера, м/с.
        dt: Шаг времени, с.

    Returns:
        np.ndarray: Нормированный вектор входа shape (3,).
    """
    raw = np.array([velocity_east, velocity_north, dt], dtype=np.float64)
    return (raw - model["input_center"]) / model["input_scale"]


def _gp_ekf_transition_stats(
    model: dict[str, object], features: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Среднее и дисперсия остатка GP-перехода и якобиан по скорости.

    Returns:
        tuple: (mean 4D, variance 4D, mean_jac_v shape (4, 2)).
    """
    diff = features - model["train_z"]
    kernel = np.exp(-0.5 * np.sum(diff * diff * model["inv_length_sq"], axis=1))
    mean_std = kernel @ model["alpha"]
    solved = solve_triangular(model["chol"], kernel, lower=True, check_finite=False)
    variance_std = (1.0 + model["noise"] ** 2) - float(solved @ solved)
    variance_std = max(variance_std, 1e-8)
    weighted = kernel[:, None] * model["alpha"]
    mean_jac_z = -((diff * model["inv_length_sq"]).T @ weighted)
    mean = model["target_center"] + model["target_scale"] * mean_std
    variance = model["target_scale"] ** 2 * variance_std
    mean_jac_v = (model["target_scale"][:, None] * mean_jac_z[:2].T) / model["input_scale"][:2]
    return mean, variance, mean_jac_v


def _gp_ekf_predict_step(
    model: dict[str, object], state: np.ndarray, cov: np.ndarray, dt: float
) -> tuple[np.ndarray, np.ndarray]:
    """Причинный шаг GP-EKF: предсказание состояния и ковариации.

    Args:
        model: Словарь обученной модели из fit_gp_ekf_on_train.
        state: Вектор состояния (x, n, v_E, v_N).
        cov: Ковариация состояния shape (4, 4).
        dt: Шаг времени, с.

    Returns:
        tuple[np.ndarray, np.ndarray]: (predicted_state, predicted_cov).
    """
    mean, variance, mean_jac_v = _gp_ekf_transition_stats(
        model, _gp_ekf_features(model, state[2], state[3], dt)
    )
    transition = np.eye(4)
    transition[0, 2] = dt
    transition[1, 3] = dt
    linear = transition.copy()
    linear[:, 2] += mean_jac_v[:, 0]
    linear[:, 3] += mean_jac_v[:, 1]
    process = np.diag(np.maximum(variance, np.array([1.0, 1.0, 1e-4, 1e-4])))
    predicted = transition @ state + mean
    predicted_cov = linear @ cov @ linear.T + process
    return predicted, 0.5 * (predicted_cov + predicted_cov.T)


def gp_ekf_nis_scores(frame: pd.DataFrame, model: dict[str, object]) -> np.ndarray:
    """NIS наблюдения GP-EKF на каждом шаге (индекс 0 — 0).

    Args:
        frame: DataFrame с колонками x, n и time.
        model: Словарь обученной модели из fit_gp_ekf_on_train.

    Returns:
        np.ndarray: NIS длины N.
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    count = len(xy)
    scores = np.zeros(count, dtype=np.float64)
    if count < 2:
        return scores
    state = np.zeros(4, dtype=np.float64)
    state[:2] = xy[0]
    cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    observation = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
    meas_cov = np.eye(2) * (GP_EKF_SIGMA_MEAS**2)
    eye = np.eye(4)
    for index in range(1, count):
        dt = float(time_sec[index] - time_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > GP_EKF_GAP_SEC
        dt_model = min(dt, GP_EKF_MAX_DT_SEC)
        pred, pred_cov = _gp_ekf_predict_step(model, state, cov, dt_model)
        innov = xy[index] - observation @ pred
        innov_cov = observation @ pred_cov @ observation.T + meas_cov
        solved = np.linalg.solve(innov_cov, innov)
        scores[index] = float(innov @ solved)
        gain = pred_cov @ observation.T @ np.linalg.inv(innov_cov)
        state = pred + gain @ innov
        updated = eye - gain @ observation
        cov = updated @ pred_cov @ updated.T + gain @ meas_cov @ gain.T
        cov = 0.5 * (cov + cov.T)
        if gap:
            state = np.array([xy[index, 0], xy[index, 1], 0.0, 0.0])
            cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    return scores


GP_EKF_CHI999 = float(chi2.ppf(0.999, 2))

gp_ekf_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

gp_ekf_rows: list[dict[str, object]] = []
gp_ekf_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in gp_ekf_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"GP-EKF: {scenario_name}")
    started = time.perf_counter()
    gp_ekf_model = fit_gp_ekf_on_train(train)
    train_scores = gp_ekf_nis_scores(train, gp_ekf_model)
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=(GP_EKF_CHI999,),
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана NIS на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  опорных переходов {gp_ekf_model['n_support']}, "
        f"ℓ {gp_ekf_model['lengthscale']:.2g}, шум {gp_ekf_model['noise']:.2g}; "
        f"χ²₀.₉₉₉={GP_EKF_CHI999:.2f}; порог NIS {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = gp_ekf_nis_scores(test, gp_ekf_model)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    gp_ekf_rows.append(test_report_row(scenario_name, metrics))
    gp_ekf_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

gp_ekf_summary = pd.DataFrame(gp_ekf_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    gp_ekf_summary.loc[:, report_columns],
    caption="GP-EKF: сводные метрики на tes)",
)

_gp_ekf_ideal_test = gp_ekf_scenario_results[0]["test"]
_gp_ekf_map_tail = min(PLOT_TAIL, len(_gp_ekf_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_gp_ekf_map_tail)} точек теста df1 (bbox: move + 5%)")
_gp_ekf_dummy_move = np.zeros(len(_gp_ekf_ideal_test), dtype=bool)
_gp_ekf_truth_pred, _gp_ekf_truth_gt = trajectory_series_pair(_gp_ekf_ideal_test, _gp_ekf_dummy_move)
_gp_ekf_truth_pred = slice_trajectory_series(_gp_ekf_truth_pred, _gp_ekf_map_tail)
_gp_ekf_truth_gt = slice_trajectory_series(_gp_ekf_truth_gt, _gp_ekf_map_tail)
plot_track_maps(
    _gp_ekf_truth_pred,
    _gp_ekf_truth_gt,
    map_panel="ground_truth",
    title="GP-EKF — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in gp_ekf_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"GP-EKF — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Скрытая марковская модель (HMM)

### Постановка

По дискретной GPS-траектории в локальной плоскости ENU строится последовательность векторных наблюдений $o_i\in\mathbb{R}^3$, $i=0,\ldots,N-1$. Компоненты $o_i=(v_i,a_i,\Delta\psi_i)$ — модуль скорости (м/с), скоростное ускорение и приращение курса, вычисленные по координатам $(x_i,n_i)$ и времени $t_i$.

На обучающей выборке по точкам штатного хода ($y_i=\mathrm{move}$) оцениваются медиана $m$ и межквартильный размах $\mathrm{IQR}$ по каждой координате; наблюдения приводятся к виду $\tilde o_i=(o_i-m)/\mathrm{IQR}$. Оценки $m$ и $\mathrm{IQR}$ фиксируются на train и применяются к test без переобучения шкалы.

Динамика ряда $\tilde o_i$ описывается скрытой марковской моделью с гауссовскими эмиссиями и диагональными ковариациями:

$$S_i\in\{1,\ldots,K\}, \quad \pi_k=P(S_1=k), \quad A_{jk}=P(S_i=k\mid S_{i-1}=j), \quad \tilde o_i\mid S_i=k \sim \mathcal N(\mu_k,\mathrm{diag}(\sigma_k^2)).$$

Оцениваются три набора параметров $\theta_N$, $\theta_A$ и при необходимости $\theta_F$. Модель $\theta_N$ обучается по непрерывным отрезкам штатного хода; $\theta_A$ — по отрезкам с $y_i=\mathrm{anomaly}$. Если на train есть отрезки anomaly с $v_i\ge v_{\mathrm{fast}}$ (в эксперименте $v_{\mathrm{fast}}=8$ м/с, константа HMM_FAST_SPEED_MPS), дополнительно оценивается $\theta_F$; иначе в детекторе используется только $\theta_A$. Непрерывность отрезка нарушается при превышении порога $\Delta t$ между соседними точками; для каждого класса в обучение включается ограниченное число самых длинных отрезков.

### Score

Для параметров $\theta$ вычисляется причинное предсказательное лог-правдоподобие $\ell_i(\theta)=\log P(\tilde o_i\mid \tilde o_{1:i-1},\theta)$.

$$s_i = \max_{k\in\{A,F\}} \ell_i(\theta_k) - \ell_i(\theta_N),$$

где член с индексом $F$ учитывается только при наличии $\theta_F$. Решение о аномалии: $\hat a_i = \mathbf{1}[s_i > \tau]$.

### Обучение и порог

Параметры $\theta_N$, $\theta_A$, $\theta_F$ оцениваются алгоритмом EM (Baum–Welch) с фиксированным числом итераций и числом скрытых состояний для каждого класса. Порог $\tau$ подбирается только на train по сетке квантилей $\{s_i\}$: выбирается кандидат с максимальным коэффициентом корреляции Мэттьюса; при равенстве в пределах полосы MCC_BAND — с минимальным операционным distance loss. На test используются зафиксированные $\theta_\cdot$, $\tau$ и та же робастная шкала; score пересчитывается по тестовой последовательности. Сценарии df1 и df2 соответствуют долям обучения TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
HMM_N_STATES_NORMAL = 3
HMM_N_STATES_ANOMALY = 3
HMM_N_STATES_FAST = 2
HMM_N_ITER = 8
HMM_MAX_SEQUENCES = 6
HMM_MAX_LEN = 1_200
HMM_MIN_LEN = 30
HMM_MIN_LEN_FAST = 20
HMM_MAX_DT_SEC = 90.0
HMM_FAST_SPEED_MPS = 8.0

def hmm_observation_matrix(frame: pd.DataFrame) -> np.ndarray:
    """Сырые наблюдения HMM: столбцы (v, a, Δψ).

    Args:
        frame: DataFrame с колонками x, n и time.

    Returns:
        np.ndarray: Матрица (N, 3) наблюдений.
    """
    speed, acc, dpsi, _ = trajectory_kinematics(frame)
    return np.column_stack([speed, acc, dpsi])

def hmm_scaled_observations(
    frame: pd.DataFrame, center: np.ndarray, scale: np.ndarray
) -> np.ndarray:
    """Наблюдения HMM после общей робастной шкалы train.

    Args:
        frame: DataFrame с колонками x, n и time.
        center: Центры из robust_params(train move).
        scale: Масштабы из robust_params(train move).

    Returns:
        np.ndarray: Нормированные наблюдения (N, 3).
    """
    return apply_robust(hmm_observation_matrix(frame), center, scale)

def hmm_class_sequence_slices(
    frame: pd.DataFrame,
    class_mask: np.ndarray,
    max_dt_sec: float,
    max_sequences: int,
    max_len: int,
    min_len: int,
) -> list[slice]:
    """Непрерывные отрезки class_mask с ограничением Δt между точками.

    Args:
        frame: DataFrame с колонками x, n и time.
        class_mask: Маска нужного класса.
        max_dt_sec: Максимальный Δt внутри отрезка, с.
        max_sequences: Максимальное число отрезков.
        max_len: Максимальная длина отрезка в точках.
        min_len: Минимальная длина отрезка в точках.

    Returns:
        list[slice]: Отрезки от длинных к коротким.
    """
    _, _, _, step_dt = trajectory_kinematics(frame)
    flag = np.asarray(class_mask, dtype=bool).copy()
    if flag.size > 1:
        flag[1:] &= step_dt[1:] <= max_dt_sec
    padded = np.r_[False, flag, False]
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    order = np.argsort(-(ends - starts))
    slices: list[slice] = []
    for index in order:
        length = int(ends[index] - starts[index])
        if length < min_len:
            break
        stop = min(int(ends[index]), int(starts[index]) + max_len)
        slices.append(slice(int(starts[index]), stop))
        if len(slices) >= max_sequences:
            break
    return slices

def hmm_gaussian_predictive_loglik(model, values: np.ndarray) -> np.ndarray:
    """Пошаговое log P(o_i | o_{1:i-1}) для обученной GaussianHMM (hmmlearn).

    Args:
        model: Обученная GaussianHMM.
        values: Нормированные наблюдения (N, 3).

    Returns:
        np.ndarray: Предсказательное логправдоподобие по точкам.
    """
    log_emit = model._compute_log_likelihood(values)
    log_start = np.log(model.startprob_ + 1e-16)
    log_trans = np.log(model.transmat_ + 1e-16)
    output = np.empty(len(values), dtype=float)
    log_filt = log_start.copy()
    for index in range(len(values)):
        log_pred = log_start if index == 0 else logsumexp(log_filt[:, None] + log_trans, axis=0)
        joint = log_pred + log_emit[index]
        output[index] = logsumexp(joint)
        log_filt = joint - output[index]
    return output

def fit_hmm_detector_on_train(train: pd.DataFrame, seed: int) -> dict[str, object]:
    """Обучает θ_N, θ_A и опционально θ_F на отрезках train (hmmlearn GaussianHMM).

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        seed: Seed инициализации HMM.

    Returns:
        dict[str, object]: center, scale, normal_model, anomaly_models и счётчики отрезков.
    """

    move_mask = train["y"].astype(str).to_numpy() == "move"
    raw_train = hmm_observation_matrix(train)
    center, scale = robust_params(raw_train[move_mask])
    values = apply_robust(raw_train, center, scale)
    labels = train["y"].astype(str).to_numpy()
    speed, _, _, _ = trajectory_kinematics(train)
    anomaly_mask = labels == "anomaly"
    fast_mask = anomaly_mask & (speed >= HMM_FAST_SPEED_MPS)
    normal_slices = hmm_class_sequence_slices(
        train, move_mask, HMM_MAX_DT_SEC, HMM_MAX_SEQUENCES, HMM_MAX_LEN, HMM_MIN_LEN
    )
    anomaly_slices = hmm_class_sequence_slices(
        train, anomaly_mask, HMM_MAX_DT_SEC, HMM_MAX_SEQUENCES, HMM_MAX_LEN, HMM_MIN_LEN
    )
    fast_slices = hmm_class_sequence_slices(
        train,
        fast_mask,
        HMM_MAX_DT_SEC,
        HMM_MAX_SEQUENCES,
        HMM_MAX_LEN,
        HMM_MIN_LEN_FAST,
    )
    if not normal_slices or not anomaly_slices:
        raise RuntimeError("Для HMM не нашлось отрезков move и anomaly на train.")
    normal_seqs = [values[sl] for sl in normal_slices]
    anomaly_seqs = [values[sl] for sl in anomaly_slices]

    def _fit_hmm(sequences: list[np.ndarray], n_states: int, model_seed: int) -> GaussianHMM:
        """Обучает GaussianHMM на наборе последовательностей.

        Args:
            sequences: Список нормированных последовательностей (n_i, 3).
            n_states: Число скрытых состояний.
            model_seed: Seed инициализации.

        Returns:
            GaussianHMM: Обученная модель.
        """
        stacked = np.concatenate(sequences, axis=0)
        lengths = [len(seq) for seq in sequences]
        hmm = GaussianHMM(
            n_components=n_states,
            covariance_type="diag",
            n_iter=HMM_N_ITER,
            random_state=model_seed,
            init_params="stmc",
            params="stmc",
        )
        hmm.fit(stacked, lengths)
        return hmm

    normal_model = _fit_hmm(normal_seqs, HMM_N_STATES_NORMAL, seed)
    anomaly_models: list[GaussianHMM] = [
        _fit_hmm(anomaly_seqs, HMM_N_STATES_ANOMALY, seed + 1)
    ]
    if fast_slices:
        fast_seqs = [values[sl] for sl in fast_slices]
        anomaly_models.append(_fit_hmm(fast_seqs, HMM_N_STATES_FAST, seed + 2))
    return {
        "center": center,
        "scale": scale,
        "normal_model": normal_model,
        "anomaly_models": anomaly_models,
        "n_normal_seqs": len(normal_seqs),
        "n_anomaly_seqs": len(anomaly_seqs),
        "n_fast_seqs": len(fast_slices),
    }

def hmm_point_anomaly_scores(frame: pd.DataFrame, detector: dict[str, object]) -> np.ndarray:
    """Score: max_k log P(o|θ_k) - log P(o|θ_N) по предсказательному правдоподобию.

    Args:
        frame: DataFrame с колонками x, n и time.
        detector: Словарь обученного детектора из fit_hmm_detector_on_train.

    Returns:
        np.ndarray: Score длины N.
    """
    values = hmm_scaled_observations(frame, detector["center"], detector["scale"])
    normal_log = hmm_gaussian_predictive_loglik(detector["normal_model"], values)
    anomaly_logs = np.vstack(
        [hmm_gaussian_predictive_loglik(model, values) for model in detector["anomaly_models"]]
    )
    return anomaly_logs.max(axis=0) - normal_log

hmm_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

hmm_rows: list[dict[str, object]] = []
hmm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in hmm_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"HMM (hmmlearn): {scenario_name}")
    started = time.perf_counter()
    detector = fit_hmm_detector_on_train(train, SEED)
    print(
        f"  отрезков move {detector['n_normal_seqs']}, anomaly {detector['n_anomaly_seqs']}, "
        f"быстрых {detector['n_fast_seqs']}; аномальных HMM {len(detector['anomaly_models'])}"
    )
    train_scores = hmm_point_anomaly_scores(train, detector)
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = hmm_point_anomaly_scores(test, detector)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    hmm_rows.append(test_report_row(scenario_name, metrics))
    hmm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

hmm_summary = pd.DataFrame(hmm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    hmm_summary.loc[:, report_columns],
    caption="HMM (hmmlearn): сводные метрики на test",
)

_hmm_ideal_test = hmm_scenario_results[0]["test"]
_hmm_map_tail = min(PLOT_TAIL, len(_hmm_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_hmm_map_tail)} точек теста df1 (bbox: move + 5%)")
_hmm_dummy_move = np.zeros(len(_hmm_ideal_test), dtype=bool)
_hmm_truth_pred, _hmm_truth_gt = trajectory_series_pair(_hmm_ideal_test, _hmm_dummy_move)
_hmm_truth_pred = slice_trajectory_series(_hmm_truth_pred, _hmm_map_tail)
_hmm_truth_gt = slice_trajectory_series(_hmm_truth_gt, _hmm_map_tail)
plot_track_maps(
    _hmm_truth_pred,
    _hmm_truth_gt,
    map_panel="ground_truth",
    title="HMM — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in hmm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"HMM — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Фильтр Хэмпела

### Постановка

По последовательности модулей скорости $v_i$ (м/с), вычисленных по координатам ENU и времени, в скользящем окне оцениваются медиана и типичный разброс (MAD). На краях ряда окно дополняется крайним значением (edge padding).

### Score

$$W_i=\{v_{i-h},\ldots,v_{i+h}\}, \quad m_i=\mathrm{med}(W_i), \quad \mathrm{MAD}_i=\mathrm{med}_{x\in W_i}|x-m_i|,$$

$$s_i=\frac{|v_i-m_i|}{1{,}4826\,\mathrm{MAD}_i+\varepsilon}, \qquad \hat a_i=\mathbf{1}[s_i>\tau].$$

Полуокно $h$ задаётся константой HAMPEL_HALF_WINDOW (окно $2h+1$ точек).

### Обучение и порог

Порог $\tau$ на train: сетка квантилей $\{s_i\}$ и ориентиры HAMPEL_THRESHOLD_EXTRA; максимум MCC; в полосе MCC_BAND — минимальный операционный distance loss. $h$ не подгоняется. На test — тот же $\tau$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
HAMPEL_HALF_WINDOW = 10
HAMPEL_THRESHOLD_EXTRA = (3.0, 3.5, 5.0)

def hampel_scores(speed_mps: np.ndarray, half_window: int) -> np.ndarray:
    """Hampel-статистика по скорости: |v - med| / (1.4826·MAD + ε), окно 2h+1.

    Args:
        speed_mps: Скорость по точкам, м/с.
        half_window: Полуразмер окна h (окно из 2h+1 точек, края — edge padding).

    Returns:
        np.ndarray: Score той же длины, что speed_mps.
    """
    speed = np.asarray(speed_mps, dtype=np.float64)
    h = int(half_window)
    if h < 1:
        raise ValueError("half_window должен быть >= 1.")
    padded = np.pad(speed, h, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, 2 * h + 1)
    center = np.median(windows, axis=1)
    mad = np.median(np.abs(windows - center[:, None]), axis=1)
    return np.abs(speed - center) / (1.4826 * mad + 1e-9)

HAMPEL_HALF_WINDOW = 10
HAMPEL_THRESHOLD_EXTRA = (3.0, 3.5, 5.0)

def hampel_speed_scores(frame: pd.DataFrame, half_window: int) -> np.ndarray:
    """Hampel-score по скорости траектории из ENU и времени.

    Args:
        frame: DataFrame с колонками x, n и time.
        half_window: Полуразмер окна h (окно из 2h+1 точек).

    Returns:
        np.ndarray: Score длины N.
    """
    speed, _, _, _ = trajectory_kinematics(frame)
    return hampel_scores(speed, half_window)

hampel_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

hampel_rows: list[dict[str, object]] = []
hampel_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in hampel_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Hampel: {scenario_name}")
    started = time.perf_counter()
    train_scores = hampel_speed_scores(train, HAMPEL_HALF_WINDOW)
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=HAMPEL_THRESHOLD_EXTRA,
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  полуокно {HAMPEL_HALF_WINDOW} (окно {2 * HAMPEL_HALF_WINDOW + 1} точек); "
        f"порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores = hampel_speed_scores(test, HAMPEL_HALF_WINDOW)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    hampel_rows.append(test_report_row(scenario_name, metrics))
    hampel_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

hampel_summary = pd.DataFrame(hampel_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    hampel_summary.loc[:, report_columns],
    caption="Hampel: сводные метрики на tesе",
)

_hampel_ideal_test = hampel_scenario_results[0]["test"]
_hampel_map_tail = min(PLOT_TAIL, len(_hampel_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_hampel_map_tail)} точек теста df1 (bbox: move + 5%)")
_hampel_dummy_move = np.zeros(len(_hampel_ideal_test), dtype=bool)
_hampel_truth_pred, _hampel_truth_gt = trajectory_series_pair(_hampel_ideal_test, _hampel_dummy_move)
_hampel_truth_pred = slice_trajectory_series(_hampel_truth_pred, _hampel_map_tail)
_hampel_truth_gt = slice_trajectory_series(_hampel_truth_gt, _hampel_map_tail)
plot_track_maps(
    _hampel_truth_pred,
    _hampel_truth_gt,
    map_panel="ground_truth",
    title="Hampel — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in hampel_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"Hampel — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## DBSCAN

### Постановка

Точки траектории представляются в пространстве признаков и кластеризуются по плотности; объекты, не попавшие в кластер (метка шума $-1$), объявляются аномалиями. На train подбирается радиус $\varepsilon$; на test кластеризация выполняется заново по тестовому облаку при той же робастной шкале и $\varepsilon$.

### Признаки и решение

$$\mathbf f_i=(\Delta x_i,\Delta y_i,v_i,a_i,\Delta\psi_i,\log\max(\mathrm{shape}_i,1)), \qquad \mathbf u_i=\frac{\mathbf f_i-m}{\mathrm{IQR}},$$

где $m$ и $\mathrm{IQR}$ оцениваются по точкам $y_i=\mathrm{move}$ на train. Core-point при $|N_\varepsilon(\mathbf u_i)|\ge \mathrm{MinPts}$. Бинарное решение: $\hat a_i=\mathbf{1}[c_i=-1]$ (отдельный порог по score на test не используется).

### Обучение и порог

Радиус $\varepsilon$ выбирается на train по квантилям $k$-расстояния ($k=$ DBSCAN_MIN_SAMPLES, сетка DBSCAN_EPS_QUANTILES): максимум MCC; в полосе MCC_BAND — минимальный операционный distance loss. DBSCAN_MIN_SAMPLES и окно формы фиксированы. Диагностический score $s_i=\max(d_k, d_{\mathrm{core}})$ вводится для анализа, но не определяет $\hat a_i$ на test. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
DBSCAN_MIN_SAMPLES = 8
DBSCAN_SHAPE_WINDOW = 6
DBSCAN_EPS_QUANTILES = (0.5, 0.7, 0.85, 0.93, 0.97)

def select_dbscan_eps(
    train: pd.DataFrame,
    features_train: np.ndarray,
    min_samples: int,
    eps_quantiles: tuple[float, ...],
    mcc_band: float,
) -> dict[str, float]:
    """Подбирает ε по k-distance на train: max MCC, полоса mcc_band — min DL.

    Args:
        train: Обучающий DataFrame.
        features_train: Нормированные признаки train.
        min_samples: min_samples DBSCAN.
        eps_quantiles: Квантили k-distance для сетки ε.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: eps, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    
    ground_truth_anomaly = train["y"].astype(str).to_numpy() == "anomaly"
    neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(features_train)
    k_distance = np.sort(neighbors.kneighbors(features_train)[0][:, -1])
    eps_grid = np.unique(np.quantile(k_distance, list(eps_quantiles)))
    records: list[dict[str, float]] = []
    for eps in eps_grid:
        labels = DBSCAN(
            eps=float(eps),
            min_samples=min_samples,
            algorithm="kd_tree",
            n_jobs=-1,
        ).fit_predict(features_train)
        predicted = labels == -1
        record = _threshold_record(float(eps), predicted, train, ground_truth_anomaly)
        record["eps"] = float(eps)
        print(
            f"  eps {eps:.3f}: доля шума {predicted.mean():.1%}, "
            f"MCC {record['mcc']:.3f}, DL {record['loss_m'] / 1000:.4f} км, F1 {record['f1']:.3f}"
        )
        records.append(record)
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора eps DBSCAN.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "eps")
    return {
        "eps": chosen["eps"],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
    }


dbscan_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

dbscan_rows: list[dict[str, object]] = []
dbscan_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in dbscan_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    move_train = train["y"].astype(str).to_numpy() == "move"

    print("=" * 72)
    print(f"DBSCAN: {scenario_name}")
    started = time.perf_counter()
    raw_train = dbscan_density_matrix(train, DBSCAN_SHAPE_WINDOW)
    center, scale = robust_params(raw_train[move_train])
    features_train = dbscan_robust_features(train, center, scale, DBSCAN_SHAPE_WINDOW)
    features_test = dbscan_robust_features(test, center, scale, DBSCAN_SHAPE_WINDOW)

    selection = select_dbscan_eps(
        train,
        features_train,
        DBSCAN_MIN_SAMPLES,
        DBSCAN_EPS_QUANTILES,
        MCC_BAND,
    )
    print(
        f"  min_samples {DBSCAN_MIN_SAMPLES}; выбран eps {selection['eps']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    dbscan_model = DBSCAN(
        eps=float(selection["eps"]),
        min_samples=DBSCAN_MIN_SAMPLES,
        algorithm="kd_tree",
        n_jobs=-1,
    )
    test_labels = dbscan_model.fit_predict(features_test)
    predicted_test = test_labels == -1
    test_scores = dbscan_isolation_scores(
        features_test, DBSCAN_MIN_SAMPLES, dbscan_model.core_sample_indices_
    )
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    noise_frac = float(np.mean(predicted_test))
    print(
        f"  тест: доля шума {noise_frac:.1%}, F1 {metrics['f1']:.4f}, "
        f"DL {metrics['distance_loss_m']:.2f} м, RMSE {metrics['rmse_m']:.2f} м "
        f"(время расчёта {elapsed:.1f} с)"
    )

    dbscan_rows.append(test_report_row(scenario_name, metrics))
    dbscan_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

dbscan_summary = pd.DataFrame(dbscan_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    dbscan_summary.loc[:, report_columns],
    caption="DBSCAN: сводные метрики на test",
)

_dbscan_ideal_test = dbscan_scenario_results[0]["test"]
_dbscan_map_tail = min(PLOT_TAIL, len(_dbscan_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_dbscan_map_tail)} точек теста df1 (bbox: move + 5%)")
_dbscan_dummy_move = np.zeros(len(_dbscan_ideal_test), dtype=bool)
_dbscan_truth_pred, _dbscan_truth_gt = trajectory_series_pair(_dbscan_ideal_test, _dbscan_dummy_move)
_dbscan_truth_pred = slice_trajectory_series(_dbscan_truth_pred, _dbscan_map_tail)
_dbscan_truth_gt = slice_trajectory_series(_dbscan_truth_gt, _dbscan_map_tail)
plot_track_maps(
    _dbscan_truth_pred,
    _dbscan_truth_gt,
    map_panel="ground_truth",
    title="DBSCAN — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in dbscan_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"DBSCAN — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

## Local Outlier Factor (LOF)

### Постановка

Сравнивается локальная плотность точки с плотностью её $k$ ближайших соседей в пространстве признаков; относительно низкая плотность соответствует аномалии. LOF пересчитывается на всём размечаемом ряде (train и test отдельно); робастная шкала признаков фиксируется по train.

### Score

$$\mathrm{LOF}_k(\mathbf u)=\frac{1}{|N_k(\mathbf u)|}\sum_{\mathbf y\in N_k(\mathbf u)}\frac{\mathrm{lrd}_k(\mathbf y)}{\mathrm{lrd}_k(\mathbf u)}, \qquad s_i=-\mathrm{LOF}_k(\mathbf u_i).$$

Признаки $\mathbf f_i=(\Delta x_i,\Delta y_i,v_i,a_i,\Delta\psi_i)$; $\mathbf u_i=(\mathbf f_i-m)/\mathrm{IQR}$ по $y_i=\mathrm{move}$ на train. Решение: $\hat a_i=\mathbf{1}[s_i>\tau]$.

### Обучение и порог

Порог $\tau$ на train: квантили $\{s_i\}$ и ориентиры LOF_THRESHOLD_EXTRA; максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. Число соседей $k=$ LOF_N_NEIGHBORS фиксировано. На test — тот же $\tau$, LOF по тестовому облаку. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
LOF_N_NEIGHBORS = 35
LOF_THRESHOLD_EXTRA = (1.5, 2.0)

def lof_feature_matrix(frame: pd.DataFrame) -> np.ndarray:
    """Признаки LOF: Δx, Δy, v, a, Δψ (без абсолютных координат).

    Args:
        frame: DataFrame с колонками x, n и time.

    Returns:
        np.ndarray: Матрица (N, 5) признаков.
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    speed, acc, dpsi, _ = trajectory_kinematics(frame)
    step_dx = np.zeros(len(east), dtype=float)
    step_dy = np.zeros(len(east), dtype=float)
    if len(east) > 1:
        step_dx[1:] = np.diff(east)
        step_dy[1:] = np.diff(north)
    return np.column_stack([step_dx, step_dy, speed, acc, dpsi])

def lof_robust_features(
    frame: pd.DataFrame, center: np.ndarray, scale: np.ndarray
) -> np.ndarray:
    """Робастно нормированные признаки LOF.

    Args:
        frame: DataFrame с колонками x, n и time.
        center: Центры из robust_params(train move).
        scale: Масштабы из robust_params(train move).

    Returns:
        np.ndarray: Матрица (N, 5) после apply_robust.
    """
    return apply_robust(lof_feature_matrix(frame), center, scale)

def lof_anomaly_scores(features: np.ndarray, n_neighbors: int, seed: int) -> np.ndarray:
    """LOF-score: -negative_outlier_factor_; больше — аномальнее.

    Args:
        features: Нормированные признаки shape (n, d).
        n_neighbors: Число соседей k.
        seed: Seed для jitter, снимающего дубликаты.

    Returns:
        np.ndarray: Score длины n.
    """
    matrix = np.asarray(features, dtype=float)
    jittered = matrix + np.random.default_rng(seed).normal(0.0, 1e-8, size=matrix.shape)
    model = LocalOutlierFactor(
        n_neighbors=int(n_neighbors),
        algorithm="kd_tree",
        n_jobs=-1,
    )
    model.fit_predict(jittered)
    return -model.negative_outlier_factor_

lof_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

lof_rows: list[dict[str, object]] = []
lof_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in lof_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    move_train = train["y"].astype(str).to_numpy() == "move"
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"LOF: {scenario_name}")
    started = time.perf_counter()
    center, scale = robust_params(lof_feature_matrix(train)[move_train])
    features_train = lof_robust_features(train, center, scale)
    train_scores = lof_anomaly_scores(features_train, LOF_N_NEIGHBORS, SEED)
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=LOF_THRESHOLD_EXTRA,
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана LOF на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  k {LOF_N_NEIGHBORS}; порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    features_test = lof_robust_features(test, center, scale)
    test_scores = lof_anomaly_scores(features_test, LOF_N_NEIGHBORS, SEED)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    lof_rows.append(test_report_row(scenario_name, metrics))
    lof_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

lof_summary = pd.DataFrame(lof_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    lof_summary.loc[:, report_columns],
    caption="LOF: сводные метрики на test",
)

_lof_ideal_test = lof_scenario_results[0]["test"]
_lof_map_tail = min(PLOT_TAIL, len(_lof_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_lof_map_tail)} точек теста df1 (bbox: move + 5%)")
_lof_dummy_move = np.zeros(len(_lof_ideal_test), dtype=bool)
_lof_truth_pred, _lof_truth_gt = trajectory_series_pair(_lof_ideal_test, _lof_dummy_move)
_lof_truth_pred = slice_trajectory_series(_lof_truth_pred, _lof_map_tail)
_lof_truth_gt = slice_trajectory_series(_lof_truth_gt, _lof_map_tail)
plot_track_maps(
    _lof_truth_pred,
    _lof_truth_gt,
    map_panel="ground_truth",
    title="LOF — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in lof_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"LOF — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

## Isolation Forest

### Постановка

Ансамбль случайных деревьев изолирует точки последовательными разрезами по признакам; объекты, отделяемые за малое число разбиений, трактуются как аномалии. Лес обучается на всей обучающей выборке; на test применяется тот же лес в том же пространстве признаков.

### Score

Нормировка глубины $c(n)=2H(n-1)-2(n-1)/n$, $n$ — размер подвыборки дерева:

$$s_i^{\mathrm{raw}}=2^{-E[h(\mathbf u_i)]/c(n)}.$$

В реализации используется $s_i=-f_{\mathrm{decision}}(\mathbf u_i)$ (большее $s_i$ — более аномальная точка). Признаки

$$\mathbf f_i=(\Delta x_i,\Delta y_i,v_i,a_i,\Delta\psi_i,\log\max(\mathrm{shape}_i,1)), \qquad \mathbf u_i=\frac{\mathbf f_i-m}{\mathrm{IQR}}$$

по $y_i=\mathrm{move}$ на train. Решение: $\hat a_i=\mathbf{1}[s_i>\tau]$.

### Обучение и порог

Порог $\tau$ на train: квантили $\{s_i\}$ и IF_THRESHOLD_EXTRA; максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. Число деревьев и размер подвыборки (IF_N_ESTIMATORS, IF_MAX_SAMPLES) и окно формы фиксированы. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
IF_N_ESTIMATORS = 200
IF_MAX_SAMPLES = 256
IF_SHAPE_WINDOW = 6
IF_THRESHOLD_EXTRA = (0.0,)

def fit_isolation_forest(
    features: np.ndarray, seed: int, n_estimators: int, max_samples: int
) -> IsolationForest:
    """Обучает Isolation Forest на всей обучающей выборке.

    Args:
        features: Нормированные признаки shape (n, d).
        seed: random_state.
        n_estimators: Число деревьев.
        max_samples: Размер подвыборки на дерево.

    Returns:
        IsolationForest: Обученная модель.
    """
    model = IsolationForest(
        n_estimators=int(n_estimators),
        max_samples=int(max_samples),
        contamination="auto",
        random_state=int(seed),
        n_jobs=-1,
    )
    model.fit(np.asarray(features, dtype=float))
    return model

def isolation_forest_scores(model: IsolationForest, features: np.ndarray) -> np.ndarray:
    """IF-score: -decision_function; больше — аномальнее.

    Args:
        model: Обученный IsolationForest.
        features: Нормированные признаки shape (n, d).

    Returns:
        np.ndarray: Score длины n.
    """
    return -model.decision_function(np.asarray(features, dtype=float))

if_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

if_rows: list[dict[str, object]] = []
if_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in if_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    move_train = train["y"].astype(str).to_numpy() == "move"
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Isolation Forest: {scenario_name}")
    started = time.perf_counter()
    raw_train = dbscan_density_matrix(train, IF_SHAPE_WINDOW)
    center, scale = robust_params(raw_train[move_train])
    features_train = isolation_forest_robust_features(train, center, scale, IF_SHAPE_WINDOW)
    forest = fit_isolation_forest(
        features_train, SEED, IF_N_ESTIMATORS, IF_MAX_SAMPLES
    )
    train_scores = isolation_forest_scores(forest, features_train)
    selection = select_anomaly_threshold(
        train,
        train_scores,
        MCC_BAND,
        extra_thresholds=IF_THRESHOLD_EXTRA,
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  {IF_N_ESTIMATORS} деревьев, max_samples {IF_MAX_SAMPLES}; "
        f"порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    features_test = isolation_forest_robust_features(test, center, scale, IF_SHAPE_WINDOW)
    test_scores = isolation_forest_scores(forest, features_test)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    if_rows.append(test_report_row(scenario_name, metrics))
    if_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

if_summary = pd.DataFrame(if_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    if_summary.loc[:, report_columns],
    caption="Isolation Forest: сводные метрики на test",
)

_if_ideal_test = if_scenario_results[0]["test"]
_if_map_tail = min(PLOT_TAIL, len(_if_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_if_map_tail)} точек теста df1 (bbox: move + 5%)")
_if_dummy_move = np.zeros(len(_if_ideal_test), dtype=bool)
_if_truth_pred, _if_truth_gt = trajectory_series_pair(_if_ideal_test, _if_dummy_move)
_if_truth_pred = slice_trajectory_series(_if_truth_pred, _if_map_tail)
_if_truth_gt = slice_trajectory_series(_if_truth_gt, _if_map_tail)
plot_track_maps(
    _if_truth_pred,
    _if_truth_gt,
    map_panel="ground_truth",
    title="Isolation Forest — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in if_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"Isolation Forest — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## One-Class SVM

### Постановка

По точкам штатного хода оценивается компактная область высокой плотности в пространстве признаков с ядром RBF. Точка вне области трактуется как аномалия. Обучение выполняется на случайной подвыборке move не более OCSVM_MAX_TRAIN_SAMPLES.

### Score и решение

$$f(\mathbf u)=\sum_j \alpha_j K(\mathbf u_j,\mathbf u)-\rho, \qquad s_i=-f(\mathbf u_i).$$

Вектор признаков $\mathbf f_i$ включает приращения, кинематику и индекс формы окна OCSVM_SHAPE_WINDOW; $\mathbf u_i=(\mathbf f_i-m)/\mathrm{IQR}$ по move на train. На train и test: $\hat a_i=\mathbf{1}[s_i>0]$ (порог score фиксирован нулём).

### Обучение и порог

Параметр $\nu$ перебирается по сетке OCSVM_NU_GRID: для каждого значения модель обучается на подвыборке move, маска на всём train при $s_i>0$; выбирается $\nu$ с максимумом MCC, в полосе MCC_BAND — минимальный операционный distance loss. На test — выбранное $\nu$ и порог $s>0$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.



In [ ]:
OCSVM_NU_GRID = (0.01, 0.05, 0.1)
OCSVM_MAX_TRAIN_SAMPLES = 4_000
OCSVM_SHAPE_WINDOW = 6

def ocsvm_anomaly_scores(model: OneClassSVM, features: np.ndarray) -> np.ndarray:
    """OC-SVM score: -decision_function; аномалия при s>0.

    Args:
        model: Обученный OneClassSVM.
        features: Нормированные признаки shape (n, d).

    Returns:
        np.ndarray: Score длины n.
    """
    return -model.decision_function(np.asarray(features, dtype=float))

def select_ocsvm_nu(
    train: pd.DataFrame,
    features_train: np.ndarray,
    features_move: np.ndarray,
    nu_grid: tuple[float, ...],
    max_train_samples: int,
    mcc_band: float,
    seed: int,
) -> dict[str, object]:
    """Подбирает параметр ν One-Class SVM по сетке nu_grid.

    Обучение — на подвыборке move; маска anomaly — при score > 0 на всём train.

    Args:
        train: Обучающий DataFrame.
        features_train: Нормированные признаки train.
        features_move: Нормированные признаки move-точек train.
        nu_grid: Кандидаты ν.
        max_train_samples: Максимум move-точек для обучения.
        mcc_band: Допуск по MCC относительно лучшего значения.
        seed: Seed подвыборки move.

    Returns:
        dict[str, object]: nu, model, train_mcc, train_f1, train_dl_m, best_mcc,
        band_size, train_sample_size.
    """
    ground_truth_anomaly = train["y"].astype(str).to_numpy() == "anomaly"
    move_matrix = np.asarray(features_move, dtype=float)
    sample_size = min(len(move_matrix), int(max_train_samples))
    sample_index = np.random.default_rng(seed).choice(
        len(move_matrix), size=sample_size, replace=False
    )
    fit_move = move_matrix[sample_index]
    records: list[dict[str, float]] = []
    models: dict[float, OneClassSVM] = {}
    train_features = np.asarray(features_train, dtype=float)
    for nu in nu_grid:
        nu_value = float(nu)
        model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu_value)
        model.fit(fit_move)
        scores = ocsvm_anomaly_scores(model, train_features)
        predicted = scores > 0.0
        record = _threshold_record(nu_value, predicted, train, ground_truth_anomaly)
        record["nu"] = nu_value
        print(
            f"  nu {nu_value:.2f}: MCC {record['mcc']:.3f}, "
            f"DL {record['loss_m'] / 1000:.4f} км, F1 {record['f1']:.3f}"
        )
        records.append(record)
        models[nu_value] = model
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора nu One-Class SVM.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "nu")
    return {
        "nu": chosen["nu"],
        "model": models[chosen["nu"]],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
        "train_sample_size": float(sample_size),
    }

ocsvm_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

ocsvm_rows: list[dict[str, object]] = []
ocsvm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in ocsvm_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    move_train = train["y"].astype(str).to_numpy() == "move"
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"One-Class SVM: {scenario_name}")
    started = time.perf_counter()
    raw_train = dbscan_density_matrix(train, OCSVM_SHAPE_WINDOW)
    center, scale = robust_params(raw_train[move_train])
    features_train = isolation_forest_robust_features(
        train, center, scale, OCSVM_SHAPE_WINDOW
    )
    features_move = apply_robust(raw_train[move_train], center, scale)
    selection = select_ocsvm_nu(
        train,
        features_train,
        features_move,
        OCSVM_NU_GRID,
        OCSVM_MAX_TRAIN_SAMPLES,
        MCC_BAND,
        SEED,
    )
    model = selection["model"]
    train_scores = ocsvm_anomaly_scores(model, features_train)
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  nu {selection['nu']:.2f}, обучение на {int(selection['train_sample_size'])} move; "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км; "
        f"порог score 0 (фикс.)"
    )

    features_test = isolation_forest_robust_features(
        test, center, scale, OCSVM_SHAPE_WINDOW
    )
    test_scores = ocsvm_anomaly_scores(model, features_test)
    predicted_test = test_scores > 0.0
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    ocsvm_rows.append(test_report_row(scenario_name, metrics))
    ocsvm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

ocsvm_summary = pd.DataFrame(ocsvm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    ocsvm_summary.loc[:, report_columns],
    caption="One-Class SVM: сводные метрики на test",
)

_ocsvm_ideal_test = ocsvm_scenario_results[0]["test"]
_ocsvm_map_tail = min(PLOT_TAIL, len(_ocsvm_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_ocsvm_map_tail)} точек теста df1 (bbox: move + 5%)")
_ocsvm_dummy_move = np.zeros(len(_ocsvm_ideal_test), dtype=bool)
_ocsvm_truth_pred, _ocsvm_truth_gt = trajectory_series_pair(_ocsvm_ideal_test, _ocsvm_dummy_move)
_ocsvm_truth_pred = slice_trajectory_series(_ocsvm_truth_pred, _ocsvm_map_tail)
_ocsvm_truth_gt = slice_trajectory_series(_ocsvm_truth_gt, _ocsvm_map_tail)
plot_track_maps(
    _ocsvm_truth_pred,
    _ocsvm_truth_gt,
    map_panel="ground_truth",
    title="One-Class SVM — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in ocsvm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"One-Class SVM — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Robust PCA (Principal Component Pursuit)

### Постановка

Матрица признаков по скользящим окнам ряда раскладывается на низкоранговую компоненту и разреженную аномальную: $X=L+S$. Разложение inexact ALM выполняется отдельно для train и для test по соответствующему ряду; порог переносится с train.

### Score

Окна длины RPCA_WINDOW_LENGTH, шаг RPCA_WINDOW_STRIDE; окно не пересекает разрыв $\Delta t>$ RPCA_MAX_DT_SEC. Строка окна — $(\Delta x/1000,\Delta y/1000,v,\Delta\psi)$; штраф $\lambda=1/\sqrt{\max(n,m)}$. Точечный score — максимум нормы разреженной компоненты по окнам, покрывающим индекс: $s_i=\max_{W\ni i}\|S_W(i)\|$.

### Обучение и порог

Порог $\tau$ на train по квантилям $s_i$: максимум MCC; в полосе MCC_BAND — минимальный операционный distance loss. На test — тот же $\tau$, новое разложение $X=L+S$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
RPCA_WINDOW_LENGTH = 16
RPCA_WINDOW_STRIDE = 8
RPCA_MAX_DT_SEC = 180.0

def soft_threshold_rpca(matrix: np.ndarray, amount: float) -> np.ndarray:
    """Оператор мягкого порога для ℓ₁-части RPCA.

    Args:
        matrix: Входная матрица.
        amount: Порог сжатия.

    Returns:
        np.ndarray: Разреженная компонента после порога.
    """
    return np.sign(matrix) * np.maximum(np.abs(matrix) - amount, 0.0)

def inexact_alm_rpca(
    data: np.ndarray, lam: float | None = None, max_iter: int = 30, tol: float = 1e-5
):
    """Inexact ALM для robust PCA: min ||L||_* + λ||S||_1 при X = L + S.

    Args:
        data: Матрица окон признаков.
        lam: Штраф ℓ₁; по умолчанию 1/√max(n,m).
        max_iter: Максимум итераций ALM.
        tol: Относительный критерий сходимости по Frobenius.

    Returns:
        tuple[np.ndarray, np.ndarray]: (low_rank L, sparse S).
    """
    work = np.asarray(data, dtype=float)
    rows, cols = work.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(rows, cols))
    spectral = np.linalg.norm(work, 2)
    mu = 1.25 / max(spectral, 1e-8)
    mu_max = mu * 1e7
    factor = 1.6
    lagrange = work / max(spectral, np.max(np.abs(work)) / lam)
    sparse = np.zeros_like(work)
    low_rank = np.zeros_like(work)
    for _ in range(max_iter):
        basis, singular, right = np.linalg.svd(work - sparse + lagrange / mu, full_matrices=False)
        low_rank = (basis * np.maximum(singular - 1.0 / mu, 0.0)) @ right
        residual = work - low_rank + lagrange / mu
        sparse = soft_threshold_rpca(residual, lam / mu)
        error = work - low_rank - sparse
        lagrange = lagrange + mu * error
        if np.linalg.norm(error, "fro") / max(np.linalg.norm(work, "fro"), 1e-8) < tol:
            break
        mu = min(mu * factor, mu_max)
    return low_rank, sparse

def rpca_point_scores(
    arrays: dict[str, np.ndarray],
    window_length: int,
    stride: int,
    max_dt_sec: float,
) -> np.ndarray:
    """Точечные RPCA-score: max нормы S по окнам, покрывающим индекс.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        window_length: Длина окна.
        stride: Шаг между окнами.
        max_dt_sec: Макс. Δt внутри окна.

    Returns:
        np.ndarray: Score на каждой точке ряда.
    """
    features = np.column_stack(
        [
            arrays["dx"] / 1000.0,
            arrays["dy"] / 1000.0,
            arrays["speed"],
            arrays["dpsi"],
        ]
    )
    starts = window_starts(arrays, window_length, max_dt_sec, np.inf, require_move=False)
    starts = starts[::stride]
    if len(starts) == 0:
        return np.zeros(arrays["count"], dtype=float)
    index = starts[:, None] + np.arange(window_length)[None, :]
    matrix = features[index].reshape(len(starts), -1)
    center, scale = robust_params(matrix)
    _, sparse = inexact_alm_rpca(apply_robust(matrix, center, scale))
    row_score = np.linalg.norm(sparse.reshape(len(starts), window_length, -1), axis=2)
    return scatter_window_max(arrays["count"], starts, row_score, window_length)

rpca_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

rpca_rows: list[dict[str, object]] = []
rpca_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in rpca_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["y"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Robust PCA: {scenario_name}")
    started = time.perf_counter()
    arrays_train = frame_window_arrays(train)
    train_scores = rpca_point_scores(
        arrays_train, RPCA_WINDOW_LENGTH, RPCA_WINDOW_STRIDE, RPCA_MAX_DT_SEC
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.median(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.median(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  окно {RPCA_WINDOW_LENGTH}, шаг {RPCA_WINDOW_STRIDE}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    arrays_test = frame_window_arrays(test)
    test_scores = rpca_point_scores(
        arrays_test, RPCA_WINDOW_LENGTH, RPCA_WINDOW_STRIDE, RPCA_MAX_DT_SEC
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    rpca_rows.append(test_report_row(scenario_name, metrics))
    rpca_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

rpca_summary = pd.DataFrame(rpca_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    rpca_summary.loc[:, report_columns],
    caption="Robust PCA: сводные метрики на test",
)

_rpca_ideal_test = rpca_scenario_results[0]["test"]
_rpca_map_tail = min(PLOT_TAIL, len(_rpca_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_rpca_map_tail)} точек теста df1 (bbox: move + 5%)")
_rpca_dummy_move = np.zeros(len(_rpca_ideal_test), dtype=bool)
_rpca_truth_pred, _rpca_truth_gt = trajectory_series_pair(_rpca_ideal_test, _rpca_dummy_move)
_rpca_truth_pred = slice_trajectory_series(_rpca_truth_pred, _rpca_map_tail)
_rpca_truth_gt = slice_trajectory_series(_rpca_truth_gt, _rpca_map_tail)
plot_track_maps(
    _rpca_truth_pred,
    _rpca_truth_gt,
    map_panel="ground_truth",
    title="Robust PCA — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in rpca_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"Robust PCA — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## LSTM-автоэнкодер (EncDec-AD)

### Постановка

Из траектории формируются окна относительных координат (км) внутри допустимых разрывов по времени и длине шага. Encoder–decoder на LSTM восстанавливает окно; обучение только на окнах, целиком лежащих на $y_i=\mathrm{move}$.

### Score

Для окна $W$ ошибка реконструкции $e_j$; после калибровки на hold-out move оцениваются $\mu_j$, $\sigma_j^2$ по компонентам:

$$s_W=\sum_j \frac{(e_j-\mu_j)^2}{\sigma_j^2}, \qquad s_i=\max_{W\ni i} s_W.$$

Длина окна LSTM_WINDOW_LENGTH; число обучающих окон ограничено N_WINDOWS; число эпох — N_EPOCHS.

### Обучение и порог

Порог $\tau$ подбирается на train по квантилям $s_i$: максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. На test — зафиксированные веса сети, калибровка $\mu$, $\sigma^2$ и $\tau$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
LSTM_WINDOW_LENGTH = 16
LSTM_MAX_DT_TRAIN_SEC = 90.0
LSTM_MAX_DT_SCORE_SEC = 120.0
LSTM_MAX_DIST_TRAIN_M = 2_000.0

class LSTMAutoencoder(nn.Module):
    """Encoder-decoder: восстановление окна из финального состояния LSTM."""

    def __init__(self, n_features: int = 2, hidden: int = 8):
        """Создаёт encoder-decoder LSTM.

        Args:
            n_features: Число признаков точки.
            hidden: Размер скрытого состояния.
        """
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.decoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, n_features)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        """Восстанавливает окно из финального состояния encoder.

        Args:
            sequence: Окно shape (batch, length, n_features).

        Returns:
            torch.Tensor: Реконструкция той же формы.
        """
        _, state = self.encoder(sequence)
        decoded, _ = self.decoder(torch.zeros_like(sequence), state)
        return self.proj(decoded)

def fit_lstm_autoencoder(
    train: pd.DataFrame,
    window_length: int,
    max_dt_sec: float,
    max_dist_m: float,
    n_windows: int,
    n_epochs: int,
    batch_size: int,
    lr: float,
    seed: int,
) -> dict[str, object]:
    """Обучает LSTM AE на move-окнах и калибрует диагональный Махаланобис.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        window_length: Длина окна в точках.
        max_dt_sec: Максимальный Δt внутри окна при обучении, с.
        max_dist_m: Максимальный шаг внутри окна при обучении, м.
        n_windows: Максимум окон обучения.
        n_epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.
        seed: Seed выбора окон.

    Returns:
        dict[str, object]: model, error_mean, error_var, window_length, n_train_windows.
    """
    arrays = frame_window_arrays(train)
    starts = window_starts(arrays, window_length, max_dt_sec, max_dist_m, require_move=True)
    starts = take_window_starts(starts, n_windows, seed)
    train_starts, calib_starts = split_window_starts(starts, seed)
    if len(calib_starts) < 10:
        calib_starts = train_starts[: max(10, len(train_starts) // 5)]
    train_windows = relative_xy_km_windows(arrays, train_starts, window_length)
    calib_windows = relative_xy_km_windows(arrays, calib_starts, window_length)
    model = LSTMAutoencoder()
    train_torch_autoencoder(model, train_windows, n_epochs, batch_size, lr)
    calib_recon = batched_model_forward(
        model,
        np.clip(calib_windows, -30.0, 30.0),
        lambda m, b: m(b),
        TORCH_INFER_BATCH,
    )
    calib_error = (calib_recon - calib_windows).reshape(len(calib_windows), -1)
    return {
        "model": model,
        "error_mean": calib_error.mean(axis=0),
        "error_var": np.maximum(calib_error.var(axis=0), 1e-6),
        "window_length": window_length,
        "n_train_windows": len(train_starts),
    }

def lstm_point_scores(
    arrays: dict[str, np.ndarray],
    model: nn.Module,
    error_mean: np.ndarray,
    error_var: np.ndarray,
    window_length: int,
    max_dt_sec: float,
) -> np.ndarray:
    """Точечные score LSTM-AE: max оконного Махаланобиса по покрывающим окнам.

    Args:
        arrays: Кэш ряда.
        model: Обученный LSTMAutoencoder.
        error_mean: Среднее остатка на калибровке.
        error_var: Дисперсии остатка.
        window_length: Длина окна.
        max_dt_sec: Макс. Δt внутри окна при скоринге.

    Returns:
        np.ndarray: Score на каждой точке.
    """
    starts = window_starts(arrays, window_length, max_dt_sec, np.inf, require_move=False)
    if len(starts) == 0:
        return np.zeros(arrays["count"], dtype=float)
    windows = relative_xy_km_windows(arrays, starts, window_length)
    recon = batched_model_forward(
        model,
        np.clip(windows, -30.0, 30.0),
        lambda m, b: m(b),
        TORCH_INFER_BATCH,
    )
    error = (recon - windows).reshape(len(starts), -1)
    window_score = diagonal_mahalanobis(error, error_mean, error_var)
    return scatter_window_max(arrays["count"], starts, window_score, window_length)

lstm_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

lstm_rows: list[dict[str, object]] = []
lstm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in lstm_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"LSTM-автоэнкодер: {scenario_name}")
    started = time.perf_counter()
    print(f"  обучение ({N_EPOCHS} эпох, до {nfmt(N_WINDOWS)} окон move)...")
    detector = fit_lstm_autoencoder(
        train,
        LSTM_WINDOW_LENGTH,
        LSTM_MAX_DT_TRAIN_SEC,
        LSTM_MAX_DIST_TRAIN_M,
        N_WINDOWS,
        N_EPOCHS,
        TORCH_BATCH_SIZE,
        TORCH_LR,
        SEED,
    )
    arrays_train = frame_window_arrays(train)
    train_scores = lstm_point_scores(
        arrays_train,
        detector["model"],
        detector["error_mean"],
        detector["error_var"],
        int(detector["window_length"]),
        LSTM_MAX_DT_SCORE_SEC,
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    print(
        f"  окон обучения {int(detector['n_train_windows'])}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    arrays_test = frame_window_arrays(test)
    test_scores = lstm_point_scores(
        arrays_test,
        detector["model"],
        detector["error_mean"],
        detector["error_var"],
        int(detector["window_length"]),
        LSTM_MAX_DT_SCORE_SEC,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    lstm_rows.append(test_report_row(scenario_name, metrics))
    lstm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

lstm_summary = pd.DataFrame(lstm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    lstm_summary.loc[:, report_columns],
    caption="LSTM-автоэнкодер: сводные метрики на test",
)

_lstm_ideal_test = lstm_scenario_results[0]["test"]
_lstm_map_tail = min(PLOT_TAIL, len(_lstm_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_lstm_map_tail)} точек теста df1 (bbox: move + 5%)")
_lstm_dummy_move = np.zeros(len(_lstm_ideal_test), dtype=bool)
_lstm_truth_pred, _lstm_truth_gt = trajectory_series_pair(_lstm_ideal_test, _lstm_dummy_move)
_lstm_truth_pred = slice_trajectory_series(_lstm_truth_pred, _lstm_map_tail)
_lstm_truth_gt = slice_trajectory_series(_lstm_truth_gt, _lstm_map_tail)
plot_track_maps(
    _lstm_truth_pred,
    _lstm_truth_gt,
    map_panel="ground_truth",
    title="LSTM-автоэнкодер — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in lstm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"LSTM-автоэнкодер — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

## Temporal Convolutional Network

### Постановка

Каузальная свёрточная сеть предсказывает следующую точку внутри окна относительных координат (км). Обучение — только на окнах штатного хода ($y_i=\mathrm{move}$) с ограничениями на $\Delta t$ и длину шага.

### Score

Остаток одношагового прогноза в окне; агрегирование — диагональный квадрат Махаланобиса по калибровке на hold-out move, score приписывается целевой точке окна длины TCN_WINDOW_LENGTH.

### Обучение и порог

Порог $\tau$ на train по квантилям score: максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. На test — та же модель и $\tau$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
TCN_WINDOW_LENGTH = 16
TCN_MAX_DT_SEC = 90.0
TCN_MAX_DIST_TRAIN_M = 2_000.0

class CausalTCN(nn.Module):
    """
    Каузальная TCN: прогноз следующей точки по контексту.
    """

    def __init__(self, n_features: int = 2, channels: int = 16, levels: int = 3, kernel: int = 3):
        """Создаёт стек каузальных свёрточных блоков.

        Args:
            n_features: Число признаков точки.
            channels: Число каналов TCN.
            levels: Число свёрточных уровней.
            kernel: Размер ядра свёртки.
        """
        super().__init__()
        blocks: list[nn.Conv1d] = []
        pads: list[int] = []
        in_channels = n_features
        dilation = 1
        for _ in range(levels):
            blocks.append(nn.Conv1d(in_channels, channels, kernel, dilation=dilation))
            pads.append((kernel - 1) * dilation)
            in_channels = channels
            dilation *= 2
        self.blocks = nn.ModuleList(blocks)
        self.pads = pads
        self.head = nn.Linear(channels, n_features)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        """Предсказывает следующую точку по контексту окна.

        Args:
            sequence: Окно shape (batch, length, n_features).

        Returns:
            torch.Tensor: Прогноз shape (batch, n_features).
        """
        hidden = sequence.transpose(1, 2)
        for conv, pad in zip(self.blocks, self.pads):
            hidden = nn.functional.pad(hidden, (pad, 0))
            hidden = torch.relu(conv(hidden))
        return self.head(hidden[:, :, -1])

def tcn_training_pairs(
    arrays: dict[str, np.ndarray],
    window_length: int,
    max_dt_sec: float,
    max_dist_m: float,
    require_move: bool,
    limit: int | None,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Старты окон, история (км) и целевой следующий шаг для TCN.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        window_length: Длина истории в точках.
        max_dt_sec: Максимальный Δt внутри окна, с.
        max_dist_m: Максимальный шаг внутри окна, м.
        require_move: Требовать move во всех точках окна.
        limit: Максимум стартов или None.
        seed: Seed выбора стартов.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: starts, history, target.
    """
    starts = window_starts(
        arrays, window_length + 1, max_dt_sec, max_dist_m, require_move=require_move
    )
    if limit is not None:
        starts = take_window_starts(starts, limit, seed)
    history = relative_xy_km_windows(arrays, starts, window_length)
    origin = arrays["xy"][starts]
    target = ((arrays["xy"][starts + window_length] - origin) / 1000.0).astype(np.float32)
    return starts, history, target

def train_tcn_predictor(
    model: nn.Module,
    features: np.ndarray,
    targets: np.ndarray,
    epochs: int,
    batch_size: int,
    lr: float,
) -> nn.Module:
    """Обучает TCN предсказывать следующую точку окна (MSE).

    Args:
        model: CausalTCN.
        features: История окон, км.
        targets: Целевой следующий шаг, км.
        epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.

    Returns:
        nn.Module: Обученная model в режиме eval.
    """
    features = np.clip(np.ascontiguousarray(features), -30.0, 30.0).astype(np.float32)
    targets = np.clip(np.ascontiguousarray(targets), -30.0, 30.0).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(features), torch.from_numpy(targets)),
        batch_size=batch_size,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(batch_x) - batch_y) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_x)
            seen += len(batch_x)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model

def fit_tcn_predictor(
    train: pd.DataFrame,
    window_length: int,
    max_dt_sec: float,
    max_dist_m: float,
    n_windows: int,
    n_epochs: int,
    batch_size: int,
    lr: float,
    seed: int,
) -> dict[str, object]:
    """Обучает CausalTCN на move-окнах и калибрует остаток одношагового прогноза.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        window_length: Длина истории в точках.
        max_dt_sec: Максимальный Δt внутри окна при обучении, с.
        max_dist_m: Максимальный шаг внутри окна при обучении, м.
        n_windows: Максимум окон обучения.
        n_epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.
        seed: Seed выбора и разбиения окон.

    Returns:
        dict[str, object]: model, residual_mean, residual_var, window_length, n_train_windows.
    """
    arrays = frame_window_arrays(train)
    pair_starts, history, target = tcn_training_pairs(
        arrays, window_length, max_dt_sec, max_dist_m, True, n_windows, seed
    )
    order = np.random.default_rng(seed).permutation(len(pair_starts))
    cut = max(1, int(len(order) * 0.85))
    train_idx, calib_idx = order[:cut], order[cut:]
    model = CausalTCN()
    train_tcn_predictor(model, history[train_idx], target[train_idx], n_epochs, batch_size, lr)
    calib_pred = batched_model_forward(
        model,
        np.clip(history[calib_idx], -30.0, 30.0),
        lambda m, b: m(b),
        TORCH_INFER_BATCH,
    )
    residual = calib_pred - target[calib_idx]
    return {
        "model": model,
        "residual_mean": residual.mean(axis=0),
        "residual_var": np.maximum(residual.var(axis=0), 1e-6),
        "window_length": window_length,
        "n_train_windows": len(train_idx),
    }

def tcn_point_scores(
    arrays: dict[str, np.ndarray],
    model: nn.Module,
    residual_mean: np.ndarray,
    residual_var: np.ndarray,
    window_length: int,
    max_dt_sec: float,
) -> np.ndarray:
    """Точечные score TCN на целевой точке окна (диаг. Махаланобис остатка).

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        model: Обученный CausalTCN.
        residual_mean: Среднее остатка на калибровке.
        residual_var: Дисперсии остатка.
        window_length: Длина истории в точках.
        max_dt_sec: Максимальный Δt внутри окна при скоринге, с.

    Returns:
        np.ndarray: Score на каждой точке.
    """
    starts, hist, truth = tcn_training_pairs(
        arrays, window_length, max_dt_sec, np.inf, False, None, 0
    )
    if len(starts) == 0:
        return np.zeros(arrays["count"], dtype=float)
    pred = batched_model_forward(
        model,
        np.clip(hist, -30.0, 30.0),
        lambda m, b: m(b),
        TORCH_INFER_BATCH,
    )
    score = diagonal_mahalanobis(pred - truth, residual_mean, residual_var)
    point_score = np.zeros(arrays["count"], dtype=float)
    np.maximum.at(point_score, starts + window_length, score)
    return point_score

tcn_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

tcn_rows: list[dict[str, object]] = []
tcn_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in tcn_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"TCN: {scenario_name}")
    started = time.perf_counter()
    print(f"  обучение ({N_EPOCHS} эпох, до {nfmt(N_WINDOWS)} окон move)...")
    detector = fit_tcn_predictor(
        train,
        TCN_WINDOW_LENGTH,
        TCN_MAX_DT_SEC,
        TCN_MAX_DIST_TRAIN_M,
        N_WINDOWS,
        N_EPOCHS,
        TORCH_BATCH_SIZE,
        TORCH_LR,
        SEED,
    )
    arrays_train = frame_window_arrays(train)
    train_scores = tcn_point_scores(
        arrays_train,
        detector["model"],
        detector["residual_mean"],
        detector["residual_var"],
        int(detector["window_length"]),
        TCN_MAX_DT_SEC,
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    print(
        f"  окон обучения {int(detector['n_train_windows'])}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    arrays_test = frame_window_arrays(test)
    test_scores = tcn_point_scores(
        arrays_test,
        detector["model"],
        detector["residual_mean"],
        detector["residual_var"],
        int(detector["window_length"]),
        TCN_MAX_DT_SEC,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    tcn_rows.append(test_report_row(scenario_name, metrics))
    tcn_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

tcn_summary = pd.DataFrame(tcn_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    tcn_summary.loc[:, report_columns],
    caption="TCN: сводные метрики на test",
)

_tcn_ideal_test = tcn_scenario_results[0]["test"]
_tcn_map_tail = min(PLOT_TAIL, len(_tcn_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_tcn_map_tail)} точек теста df1 (bbox: move + 5%)")
_tcn_dummy_move = np.zeros(len(_tcn_ideal_test), dtype=bool)
_tcn_truth_pred, _tcn_truth_gt = trajectory_series_pair(_tcn_ideal_test, _tcn_dummy_move)
_tcn_truth_pred = slice_trajectory_series(_tcn_truth_pred, _tcn_map_tail)
_tcn_truth_gt = slice_trajectory_series(_tcn_truth_gt, _tcn_map_tail)
plot_track_maps(
    _tcn_truth_pred,
    _tcn_truth_gt,
    map_panel="ground_truth",
    title="TCN — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in tcn_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"TCN — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## DONUT (вариационный автоэнкодер)

### Постановка

Вариационный автоэнкодер обучается восстанавливать окна относительных координат (км); оптимизация — стохастическая вариационная выводка (ELBO). Обучение только на окнах штатного хода. Постановка адаптирует идею DONUT (Xu et al., WWW 2018) к одномерной траектории в плоскости ENU.

### Score

$$s_W=\|x-\hat x\|_2^2 + D_{\mathrm{KL}}\bigl(q_\phi(z|x)\|p(z)\bigr), \qquad s_i=\max_{W\ni i}s_W.$$

Параметры окна и латентного слоя задаются DONUT_WINDOW_LENGTH, DONUT_LATENT_DIM; обучение ограничено N_WINDOWS окнами и N_EPOCHS эпохами.

### Обучение и порог

Порог $\tau$ на train по квантилям $s_i$: максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. На test — зафиксированные параметры VAE и $\tau$. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
DONUT_WINDOW_LENGTH = 16
DONUT_LATENT_DIM = 8
DONUT_OBS_SIGMA = 1.0
DONUT_MAX_DT_TRAIN_SEC = 90.0
DONUT_MAX_DT_SCORE_SEC = 120.0
DONUT_MAX_DIST_TRAIN_M = 2_000.0

class DonutWindowVAE(nn.Module):
    """VAE для плоского окна (DONUT-адаптация): encoder/decoder на PyTorch."""

    def __init__(self, dim: int, latent: int = 8):
        """Создаёт encoder и decoder VAE для плоского окна.

        Args:
            dim: Размер плоского окна.
            latent: Размерность латентного пространства.
        """
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU()
        )
        self.mu_layer = nn.Linear(32, latent)
        self.logvar_layer = nn.Linear(32, latent)
        self.decoder = nn.Sequential(
            nn.Linear(latent, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim),
        )

    def encode(self, flat: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Кодирует плоское окно в параметры латентного Гаусса.

        Args:
            flat: Плоское окно shape (batch, dim).

        Returns:
            tuple[torch.Tensor, torch.Tensor]: (mu, logvar).
        """
        hidden = self.encoder(flat)
        mu = self.mu_layer(hidden)
        logvar = self.logvar_layer(hidden).clamp(-8.0, 8.0)
        return mu, logvar

    def decode(self, latent: torch.Tensor) -> torch.Tensor:
        """Восстанавливает плоское окно из латентного вектора.

        Args:
            latent: Латентный вектор shape (batch, latent).

        Returns:
            torch.Tensor: Реконструкция shape (batch, dim).
        """
        return self.decoder(latent)

    def forward(
        self, flat: torch.Tensor, sample_latent: bool
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Прогоняет окно через VAE.

        Args:
            flat: Плоское окно shape (batch, dim).
            sample_latent: Сэмплировать z вместо использования mu.

        Returns:
            tuple[torch.Tensor, torch.Tensor, torch.Tensor]: (recon, mu, logvar).
        """
        mu, logvar = self.encode(flat)
        if sample_latent:
            latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            latent = mu
        return self.decode(latent), mu, logvar

def build_donut_pyro_svi(model: DonutWindowVAE, obs_sigma: float, lr: float) -> SVI:
    """Pyro SVI: generative VAE + guide (Trace_ELBO)."""

    def model_fn(x: torch.Tensor) -> None:
        """Генеративная модель Pyro: z ~ q(z|x), obs ~ N(decode(z), σ).

        Args:
            x: Плоское окно shape (batch, dim).
        """
        pyro.module("donut_vae", model)
        with pyro.plate("data", x.shape[0]):
            mu, logvar = model.encode(x)
            std = torch.exp(0.5 * logvar)
            latent = pyro.sample("z", dist.Normal(mu, std).to_event(1))
            loc = model.decode(latent)
            pyro.sample("obs", dist.Normal(loc, obs_sigma).to_event(1), obs=x)

    def guide_fn(x: torch.Tensor) -> None:
        """Гайд Pyro: вариационное распределение z по encoder.

        Args:
            x: Плоское окно shape (batch, dim).
        """
        pyro.module("donut_vae", model)
        with pyro.plate("data", x.shape[0]):
            mu, logvar = model.encode(x)
            std = torch.exp(0.5 * logvar)
            pyro.sample("z", dist.Normal(mu, std).to_event(1))

    return SVI(model_fn, guide_fn, PyroOptim(torch.optim.Adam, {"lr": lr}), loss=Trace_ELBO())

def train_donut_vae_svi(
    model: DonutWindowVAE,
    train_table: np.ndarray,
    epochs: int,
    batch_size: int,
    lr: float,
    obs_sigma: float,
) -> DonutWindowVAE:
    """Обучение DONUT-VAE через Pyro SVI (библиотечный Trace_ELBO).

    Args:
        model: DonutWindowVAE.
        train_table: Нормированные плоские окна.
        epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.
        obs_sigma: σ наблюдения.

    Returns:
        DonutWindowVAE: Обученная модель в режиме eval.
    """
    pyro.clear_param_store()
    model.to(DEVICE)
    svi = build_donut_pyro_svi(model, obs_sigma, lr)
    array = np.ascontiguousarray(train_table).astype(np.float32)
    for epoch in range(epochs):
        order = np.random.permutation(len(array))
        total, seen = 0.0, 0
        for start in range(0, len(order), batch_size):
            idx = order[start : start + batch_size]
            batch = torch.from_numpy(array[idx]).to(DEVICE)
            loss = svi.step(batch)
            total += float(loss) * len(idx)
            seen += len(idx)
        print(f"    эпоха {epoch + 1}/{epochs}, ELBO loss {total / max(seen, 1):.5f}")
    model.eval()
    return model

def donut_vae_window_table(
    arrays: dict[str, np.ndarray],
    starts: np.ndarray,
    window_length: int,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    """Нормированная таблица плоских окон для DONUT-VAE.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        starts: Индексы начала окон.
        window_length: Длина окна в точках.
        mean: Среднее признаков на обучении.
        std: Масштаб признаков на обучении.

    Returns:
        np.ndarray: Таблица shape (n_windows, window_length * 2), float32.
    """
    windows = relative_xy_km_windows(arrays, starts, window_length).reshape(len(starts), -1)
    scaled = (windows - mean) / std
    return np.clip(scaled, -30.0, 30.0).astype(np.float32)

def donut_window_scores_batch(model: DonutWindowVAE, batch: torch.Tensor) -> torch.Tensor:
    """Оконный score: MSE реконструкции + KL(q||p), torch.distributions.

    Args:
        model: DonutWindowVAE.
        batch: Плоские окна shape (batch, dim).

    Returns:
        torch.Tensor: Score на окно shape (batch,).
    """
    recon, mu, logvar = model(batch, sample_latent=False)
    mse = ((recon - batch) ** 2).sum(dim=-1)
    std = torch.exp(0.5 * logvar)
    kl = kl_divergence(
        Normal(mu, std),
        Normal(torch.zeros_like(mu), torch.ones_like(mu)),
    ).sum(dim=-1)
    return mse + kl

def fit_donut_detector(
    train: pd.DataFrame,
    window_length: int,
    max_dt_sec: float,
    max_dist_m: float,
    n_windows: int,
    n_epochs: int,
    batch_size: int,
    lr: float,
    latent_dim: int,
    obs_sigma: float,
    seed: int,
) -> dict[str, object]:
    """Обучает DONUT-VAE (Pyro SVI) на move-окнах.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        window_length: Длина окна в точках.
        max_dt_sec: Максимальный Δt внутри окна при обучении, с.
        max_dist_m: Максимальный шаг внутри окна при обучении, м.
        n_windows: Максимум окон обучения.
        n_epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.
        latent_dim: Размерность латентного пространства.
        obs_sigma: σ наблюдения.
        seed: Seed выбора окон.

    Returns:
        dict[str, object]: model, vae_mean, vae_std, window_length, n_train_windows.
    """
    arrays = frame_window_arrays(train)
    starts = window_starts(arrays, window_length, max_dt_sec, max_dist_m, require_move=True)
    starts = take_window_starts(starts, n_windows, seed)
    raw = relative_xy_km_windows(arrays, starts, window_length).reshape(len(starts), -1)
    vae_mean = raw.mean(axis=0)
    vae_std = np.where(raw.std(axis=0) < 1e-6, 1.0, raw.std(axis=0))
    train_table = np.clip((raw - vae_mean) / vae_std, -8.0, 8.0).astype(np.float32)
    model = DonutWindowVAE(train_table.shape[1], latent_dim)
    train_donut_vae_svi(model, train_table, n_epochs, batch_size, lr, obs_sigma)
    return {
        "model": model,
        "vae_mean": vae_mean,
        "vae_std": vae_std,
        "window_length": window_length,
        "n_train_windows": len(starts),
    }

def donut_point_scores(
    arrays: dict[str, np.ndarray],
    detector: dict[str, object],
    max_dt_sec: float,
) -> np.ndarray:
    """Точечные DONUT-score: max оконного ELBO-остатка по покрывающим окнам.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        detector: Словарь обученного детектора из fit_donut_detector.
        max_dt_sec: Максимальный Δt внутри окна при скоринге, с.

    Returns:
        np.ndarray: Score на каждой точке.
    """
    window_length = int(detector["window_length"])
    starts = window_starts(arrays, window_length, max_dt_sec, np.inf, require_move=False)
    if len(starts) == 0:
        return np.zeros(arrays["count"], dtype=float)
    table = donut_vae_window_table(
        arrays,
        starts,
        window_length,
        detector["vae_mean"],
        detector["vae_std"],
    )
    model = detector["model"]
    window_score = batched_model_forward(
        model,
        table,
        donut_window_scores_batch,
        TORCH_INFER_BATCH,
    )
    return scatter_window_max(arrays["count"], starts, window_score, window_length)

donut_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

donut_rows: list[dict[str, object]] = []
donut_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in donut_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"DONUT: {scenario_name}")
    started = time.perf_counter()
    print(
        f"  обучение Pyro SVI ({N_EPOCHS} эпох, до {nfmt(N_WINDOWS)} окон move, "
        f"латентность {DONUT_LATENT_DIM})..."
    )
    detector = fit_donut_detector(
        train,
        DONUT_WINDOW_LENGTH,
        DONUT_MAX_DT_TRAIN_SEC,
        DONUT_MAX_DIST_TRAIN_M,
        N_WINDOWS,
        N_EPOCHS,
        TORCH_BATCH_SIZE,
        TORCH_LR,
        DONUT_LATENT_DIM,
        DONUT_OBS_SIGMA,
        SEED,
    )
    arrays_train = frame_window_arrays(train)
    train_scores = donut_point_scores(arrays_train, detector, DONUT_MAX_DT_SCORE_SEC)
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    print(
        f"  окон обучения {int(detector['n_train_windows'])}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    arrays_test = frame_window_arrays(test)
    test_scores = donut_point_scores(arrays_test, detector, DONUT_MAX_DT_SCORE_SEC)
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    donut_rows.append(test_report_row(scenario_name, metrics))
    donut_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

donut_summary = pd.DataFrame(donut_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    donut_summary.loc[:, report_columns],
    caption="DONUT: сводные метрики на test",
)

_donut_ideal_test = donut_scenario_results[0]["test"]
_donut_map_tail = min(PLOT_TAIL, len(_donut_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_donut_map_tail)} точек теста df1 (bbox: move + 5%)")
_donut_dummy_move = np.zeros(len(_donut_ideal_test), dtype=bool)
_donut_truth_pred, _donut_truth_gt = trajectory_series_pair(_donut_ideal_test, _donut_dummy_move)
_donut_truth_pred = slice_trajectory_series(_donut_truth_pred, _donut_map_tail)
_donut_truth_gt = slice_trajectory_series(_donut_truth_gt, _donut_map_tail)
plot_track_maps(
    _donut_truth_pred,
    _donut_truth_gt,
    map_panel="ground_truth",
    title="DONUT — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in donut_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"DONUT — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## STGVAD (spatio-temporal graph neural network)

### Постановка

В каждом окне траектории строится направленный граф: временные рёбра «прошлое → настоящее» и пространственные рёбра между точками с расстоянием не более STGVAD_EDGE_RADIUS_M. Графовая сеть предсказывает относительное положение точек в км; обучение на окнах $y_i=\mathrm{move}$.

### Score

$$s_{i,W}=1000\|\hat{\mathbf{r}}_i-\mathbf{r}_i\|_2 \quad (\text{м}), \qquad s_i=\max_{W\ni i} s_{i,W},$$

с нулевым вкладом для первой точки окна при агрегировании. Длина окна STGVAD_WINDOW_LENGTH; при оценке на ряде используется шаг STGVAD_SCORE_START_STRIDE по стартам окон.

### Обучение и порог

Порог $\tau$ на train по квантилям $s_i$: максимум MCC; полоса MCC_BAND; минимальный операционный distance loss. Число окон и эпох ограничено N_WINDOWS и N_EPOCHS. Сценарии df1/df2 — TEST_SPLIT_1 и TEST_SPLIT_2.

In [ ]:
STGVAD_WINDOW_LENGTH = 16
STGVAD_MAX_DT_TRAIN_SEC = 90.0
STGVAD_MAX_DT_SCORE_SEC = 120.0
STGVAD_MAX_DIST_TRAIN_M = 2_000.0
STGVAD_EDGE_RADIUS_M = 5_000.0
STGVAD_SCORE_START_STRIDE = 2

class STGNN(nn.Module):
    """Два слоя message passing (STGVAD); координаты вершины в own не входят."""

    def __init__(self) -> None:
        """Создаёт два слоя message passing STGVAD."""
        super().__init__()
        self.own1 = nn.Linear(2, 32)
        self.msg1 = nn.Linear(4, 32)
        self.own2 = nn.Linear(32, 32)
        self.msg2 = nn.Linear(32, 32)
        self.head = nn.Linear(32, 2)

    def forward(
        self,
        own: torch.Tensor,
        neigh: torch.Tensor,
        adj: torch.Tensor,
    ) -> torch.Tensor:
        """Прогоняет окно графа через два слоя message passing.

        Args:
            own: Признаки вершин shape (batch, length, 2).
            neigh: Признаки соседей shape (batch, length, 4).
            adj: Нормированная смежность shape (batch, length, length).

        Returns:
            torch.Tensor: Прогноз положения вершин shape (batch, length, 2).
        """
        hidden = torch.relu(self.own1(own) + self.msg1(torch.matmul(adj, neigh)))
        hidden = torch.relu(self.own2(hidden) + self.msg2(torch.matmul(adj, hidden)))
        return self.head(hidden)

def stgvad_graph_batch(
    arrays: dict[str, np.ndarray],
    starts: np.ndarray,
    length: int,
    edge_radius_m: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Признаки вершин, соседей и нормированная смежность для окон графа.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        starts: Индексы начала окон.
        length: Длина окна в точках.
        edge_radius_m: Радиус ребра графа, м.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: (own, neigh, adj, target).
    """
    index = starts[:, None] + np.arange(length)[None, :]
    xy = arrays["xy"][index]
    relative = xy - xy[:, :1, :]
    distance = np.linalg.norm(relative[:, :, None, :] - relative[:, None, :, :], axis=-1)
    past = np.tri(length, k=-1, dtype=np.float32)
    adj = ((distance < edge_radius_m) & (distance > 0.0)).astype(np.float32) * past
    adj /= np.maximum(adj.sum(axis=-1, keepdims=True), 1.0)
    speed = (arrays["speed"][index] / 10.0).astype(np.float32)
    turn = arrays["dpsi"][index].astype(np.float32)
    own = np.clip(np.stack([speed, turn], axis=-1), -30.0, 30.0).astype(np.float32)
    neigh = np.clip(
        np.stack(
            [relative[:, :, 0] / 1000.0, relative[:, :, 1] / 1000.0, speed, turn],
            axis=-1,
        ),
        -30.0,
        30.0,
    ).astype(np.float32)
    target = (relative / 1000.0).astype(np.float32)
    return own, neigh, adj.astype(np.float32), target

def train_stgvad_gnn(
    model: STGNN,
    own: np.ndarray,
    neigh: np.ndarray,
    adj: np.ndarray,
    target: np.ndarray,
    epochs: int,
    batch_size: int,
    lr: float,
) -> STGNN:
    """MSE прогноза положения (км), первая точка окна не входит в loss.

    Args:
        model: STGNN.
        own: Признаки вершин.
        neigh: Признаки соседей.
        adj: Нормированная смежность.
        target: Целевые относительные координаты, км.
        epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.

    Returns:
        STGNN: Обученная модель в режиме eval.
    """
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(own),
            torch.from_numpy(neigh),
            torch.from_numpy(adj),
            torch.from_numpy(np.clip(target, -30.0, 30.0)),
        ),
        batch_size=batch_size,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_own, batch_neigh, batch_adj, batch_y in loader:
            batch_own = batch_own.to(DEVICE)
            batch_neigh = batch_neigh.to(DEVICE)
            batch_adj = batch_adj.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(batch_own, batch_neigh, batch_adj)
            loss = ((pred[:, 1:, :] - batch_y[:, 1:, :]) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_own)
            seen += len(batch_own)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model

def fit_stgvad_detector(
    train: pd.DataFrame,
    window_length: int,
    max_dt_sec: float,
    max_dist_m: float,
    edge_radius_m: float,
    n_windows: int,
    n_epochs: int,
    batch_size: int,
    lr: float,
    seed: int,
) -> dict[str, object]:
    """Обучает STGNN на move-окнах графа траектории.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        window_length: Длина окна в точках.
        max_dt_sec: Максимальный Δt внутри окна при обучении, с.
        max_dist_m: Максимальный шаг внутри окна при обучении, м.
        edge_radius_m: Радиус ребра графа, м.
        n_windows: Максимум окон обучения.
        n_epochs: Число эпох.
        batch_size: Размер пакета.
        lr: Скорость обучения Adam.
        seed: Seed выбора окон.

    Returns:
        dict[str, object]: model, window_length, edge_radius_m, n_train_windows.
    """
    arrays = frame_window_arrays(train)
    starts = window_starts(arrays, window_length, max_dt_sec, max_dist_m, require_move=True)
    starts = take_window_starts(starts, n_windows, seed)
    own, neigh, adj, target = stgvad_graph_batch(arrays, starts, window_length, edge_radius_m)
    model = train_stgvad_gnn(STGNN(), own, neigh, adj, target, n_epochs, batch_size, lr)
    return {
        "model": model,
        "window_length": window_length,
        "edge_radius_m": edge_radius_m,
        "n_train_windows": len(starts),
    }

def stgvad_point_scores(
    arrays: dict[str, np.ndarray],
    detector: dict[str, object],
    max_dt_sec: float,
    start_stride: int,
) -> np.ndarray:
    """Точечные STGVAD-score: max ошибки прогноза (м) по окнам графа.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        detector: Словарь обученного детектора из fit_stgvad_detector.
        max_dt_sec: Максимальный Δt внутри окна при скоринге, с.
        start_stride: Шаг прореживания стартов окон.

    Returns:
        np.ndarray: Score на каждой точке.
    """
    window_length = int(detector["window_length"])
    edge_radius_m = float(detector["edge_radius_m"])
    starts = window_starts(arrays, window_length, max_dt_sec, np.inf, require_move=False)
    if start_stride > 1:
        starts = starts[::start_stride]
    if len(starts) == 0:
        return np.zeros(arrays["count"], dtype=float)
    model = detector["model"]
    model.eval()
    pieces: list[np.ndarray] = []
    for cursor in range(0, len(starts), TORCH_INFER_BATCH):
        batch_starts = starts[cursor : cursor + TORCH_INFER_BATCH]
        own, neigh, adj, target = stgvad_graph_batch(
            arrays, batch_starts, window_length, edge_radius_m
        )
        with torch.no_grad():
            pred = (
                model(
                    torch.from_numpy(own).to(DEVICE),
                    torch.from_numpy(neigh).to(DEVICE),
                    torch.from_numpy(adj).to(DEVICE),
                )
                .cpu()
                .numpy()
            )
        error = np.linalg.norm(pred - target, axis=-1) * 1000.0
        error[:, 0] = 0.0
        pieces.append(error)
    return scatter_window_max(arrays["count"], starts, np.concatenate(pieces, axis=0), window_length)

stgvad_scenarios = (
    (f"df1 (rho={TEST_SPLIT_1:.0%})", df1),
    (f"df2 (rho={TEST_SPLIT_2:.0%})", df2),
)

stgvad_rows: list[dict[str, object]] = []
stgvad_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in stgvad_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"STGVAD: {scenario_name}")
    started = time.perf_counter()
    print(
        f"  обучение STGNN ({N_EPOCHS} эпох, до {nfmt(N_WINDOWS)} окон move, "
        f"радиус ребра {STGVAD_EDGE_RADIUS_M / 1000:.0f} км)..."
    )
    detector = fit_stgvad_detector(
        train,
        STGVAD_WINDOW_LENGTH,
        STGVAD_MAX_DT_TRAIN_SEC,
        STGVAD_MAX_DIST_TRAIN_M,
        STGVAD_EDGE_RADIUS_M,
        N_WINDOWS,
        N_EPOCHS,
        TORCH_BATCH_SIZE,
        TORCH_LR,
        SEED,
    )
    arrays_train = frame_window_arrays(train)
    train_scores = stgvad_point_scores(
        arrays_train,
        detector,
        STGVAD_MAX_DT_SCORE_SEC,
        STGVAD_SCORE_START_STRIDE,
    )
    selection = select_anomaly_threshold(train, train_scores, MCC_BAND)
    print(
        f"  окон обучения {int(detector['n_train_windows'])}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    arrays_test = frame_window_arrays(test)
    test_scores = stgvad_point_scores(
        arrays_test,
        detector,
        STGVAD_MAX_DT_SCORE_SEC,
        STGVAD_SCORE_START_STRIDE,
    )
    predicted_test = test_scores > selection["tau"]
    metrics = evaluate_test(test, predicted_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м (время расчёта {elapsed:.1f} с)"
    )

    stgvad_rows.append(test_report_row(scenario_name, metrics))
    stgvad_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "metrics": metrics,
        }
    )

stgvad_summary = pd.DataFrame(stgvad_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    stgvad_summary.loc[:, report_columns],
    caption="STGVAD: сводные метрики на test",
)

_stgvad_ideal_test = stgvad_scenario_results[0]["test"]
_stgvad_map_tail = min(PLOT_TAIL, len(_stgvad_ideal_test))
print(f"\nИдеальная разметка: последние {nfmt(_stgvad_map_tail)} точек теста df1 (bbox: move + 5%)")
_stgvad_dummy_move = np.zeros(len(_stgvad_ideal_test), dtype=bool)
_stgvad_truth_pred, _stgvad_truth_gt = trajectory_series_pair(_stgvad_ideal_test, _stgvad_dummy_move)
_stgvad_truth_pred = slice_trajectory_series(_stgvad_truth_pred, _stgvad_map_tail)
_stgvad_truth_gt = slice_trajectory_series(_stgvad_truth_gt, _stgvad_map_tail)
plot_track_maps(
    _stgvad_truth_pred,
    _stgvad_truth_gt,
    map_panel="ground_truth",
    title="STGVAD — идеальная разметка",
    figsize=MAP_FIGSIZE,
    annotate=False,
)

for item in stgvad_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    print(f"\n{item['name']}")
    display_test_report_table(
        pd.DataFrame([test_report_row(str(item["name"]), item["metrics"])]).loc[:, report_columns],
        caption=f"Метрики test: {item['name']}",
    )
    print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
    prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"])
    prediction = slice_trajectory_series(prediction, tail)
    ground_truth = slice_trajectory_series(ground_truth, tail)
    plot_track_maps(
        prediction,
        ground_truth,
        map_panel="model",
        title=f"STGVAD — {item['name']}",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )



## Общая сводка по моделям



In [ ]:
def _model_summary_catalog() -> list[tuple[str, str]]:
    """Порядок строк в общей сводке: (название модели, имя DataFrame в kernel).

    Returns:
        list[tuple[str, str]]: Пары (название модели, имя переменной-сводки).
    """
    return [
        ("MPCT", "mpct_summary"),
        ("AR(p)", "ar_summary"),
        ("ARIMA(2,1,1)", "summary"),
        ("Фильтр Калмана", "kalman_summary"),
        ("Gaussian process", "gp_summary"),
        ("GP-EKF", "gp_ekf_summary"),
        ("HMM", "hmm_summary"),
        ("Hampel", "hampel_summary"),
        ("DBSCAN", "dbscan_summary"),
        ("LOF", "lof_summary"),
        ("Isolation Forest", "if_summary"),
        ("One-Class SVM", "ocsvm_summary"),
        ("Robust PCA", "rpca_summary"),
        ("LSTM-автоэнкодер", "lstm_summary"),
        ("TCN", "tcn_summary"),
        ("DONUT", "donut_summary"),
        ("STGVAD", "stgvad_summary"),
    ]


def build_master_test_table(scenario_index: int) -> pd.DataFrame:
    """Сводная таблица test для одного сценария (0 — df1, 1 — df2).

    Args:
        scenario_index: Индекс сценария в сводках моделей.

    Returns:
        pd.DataFrame: Строки «Модель» + метрики test.
    """
    rows: list[dict[str, object]] = []
    for model_name, var_name in _model_summary_catalog():
        table = globals().get(var_name)
        if not isinstance(table, pd.DataFrame) or len(table) <= scenario_index:
            continue
        summary_row = table.iloc[scenario_index]
        row: dict[str, object] = {"Модель": model_name}
        for key, _ in TEST_REPORT_METRICS:
            row[key] = summary_row.get(key, float("nan"))
        rows.append(row)
    return pd.DataFrame(rows)


def display_master_test_table(frame: pd.DataFrame, caption: str) -> None:
    """Таблица сводки: колонка «Модель» и метрики test с форматированием fmt_num.

    Args:
        frame: Таблица из build_master_test_table.
        caption: Подпись таблицы.
    """
    metric_labels = {key: label for key, label in TEST_REPORT_METRICS}
    renamed = frame.rename(columns=metric_labels)
    for key, label in TEST_REPORT_METRICS:
        if label in renamed.columns:
            renamed[label] = renamed[label].map(lambda v, k=key: _format_metric_value(k, v))
    styled = renamed.style
    if caption:
        styled = styled.set_caption(caption)
    display(styled)


_master_scenarios = (
    (
        0,
        f"df1 — доля обучения ρ={TEST_SPLIT_1:.0%} "
        f"(test {nfmt(len(df1['test']))} точек)",
    ),
    (
        1,
        f"df2 — доля обучения ρ={TEST_SPLIT_2:.0%} "
        f"(test {nfmt(len(df2['test']))} точек)",
    ),
)

for scenario_index, scenario_caption in _master_scenarios:
    master_table = build_master_test_table(scenario_index)
    print("=" * 72)
    print(scenario_caption)
    if master_table.empty:
        print("  (нет данных — выполните ячейки моделей выше)")
        continue
    display_master_test_table(
        master_table,
        caption=f"Метрики test, все модели: {scenario_caption}",
    )


## Сводка: сортировка по F1 и Distance loss

Обе итоговые таблицы test (df1 и df2): сначала строки моделей по убыванию F1, затем по возрастанию operational distance loss (DL).



In [ ]:
_master_sort_specs = (
    ("f1", False, "F1 (убывание)"),
    ("distance_loss_m", True, "Distance loss (возрастание)"),
)

for sort_key, ascending, sort_title in _master_sort_specs:
    print("=" * 72)
    print(f"Общая сводка — сортировка: {sort_title}")
    for scenario_index, scenario_caption in _master_scenarios:
        master_table = build_master_test_table(scenario_index)
        print("=" * 72)
        print(scenario_caption)
        if master_table.empty:
            print("  (нет данных — выполните ячейки моделей выше)")
            continue
        sorted_table = master_table.sort_values(
            sort_key, ascending=ascending, na_position="last"
        ).reset_index(drop=True)
        display_master_test_table(
            sorted_table,
            caption=f"{sort_title}: {scenario_caption}",
        )



## Итоговые таблицы test (сортировка)

Снимок по результатам ячейки «Общая сводка по моделям». Сначала оба сценария по убыванию F1, затем оба сценария по возрастанию Distance loss.


### Сортировка: F1


#### df1 — доля обучения ρ=60% (test 86 263 точек)


| Модель | RMSE, м | Max(d_i), м | Distance loss, м | Delay median, с | Delay mean, с | Delay p95, с | Delay max, с | On-time episodes | Late/missed episodes | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MPCT | 73_201.74 | 380_633.64 | 2_764_635.44 | 10.00 | 91.82 | 420.00 | 720.00 | 18 | 11 | 0.8527 | 0.8371 | 0.8448 |
| One-Class SVM | 135_586.87 | 927_706.26 | 24_177_277.58 | 60.00 | 86.00 | 222.00 | 260.00 | 24 | 5 | 0.7611 | 0.8389 | 0.7981 |
| Isolation Forest | 44_337.01 | 472_436.06 | 11_422_169.28 | 110.00 | 110.00 | 110.00 | 110.00 | 28 | 1 | 0.6827 | 0.9025 | 0.7774 |
| Фильтр Калмана | 63_605.52 | 456_336.43 | 8_104_878.95 | 65.00 | 995.00 | 3_290.5 | 3_850 | 25 | 4 | 0.8095 | 0.7094 | 0.7562 |
| HMM | 173_539.13 | 1_032_210.33 | 35_757_396.53 | 10.00 | 10.00 | 19.00 | 20.00 | 26 | 3 | 0.6544 | 0.7816 | 0.7124 |
| DONUT | 192_551.17 | 6_380_999.81 | 89_189_301.72 | 770.00 | 6_358.15 | 28_739 | 33_552 | 16 | 13 | 0.9553 | 0.5671 | 0.7117 |
| LSTM-автоэнкодер | 171_426.13 | 6_380_999.81 | 86_266_364.98 | 930.50 | 6_863 | 29_140 | 33_552 | 17 | 12 | 0.8709 | 0.5815 | 0.6974 |
| STGVAD | 213_800.55 | 6_380_999.81 | 124_870_896.61 | 45.00 | 1_708.5 | 5_796.3 | 25_552 | 11 | 18 | 0.7970 | 0.5111 | 0.6228 |
| Robust PCA | 158_451.27 | 6_380_999.81 | 117_428_236.07 | 30.00 | 3_217.75 | 16_626.3 | 25_552 | 21 | 8 | 0.4646 | 0.8651 | 0.6045 |
| GP-EKF | 141_435 | 1_032_210.33 | 22_281_191.9 | 100.00 | 846.00 | 3_223 | 4_000 | 24 | 5 | 0.6140 | 0.5728 | 0.5927 |
| TCN | 224_256.21 | 6_380_999.81 | 116_632_211.55 | 170.00 | 2_519.59 | 17_159.2 | 25_752 | 12 | 17 | 0.9078 | 0.4306 | 0.5841 |
| Gaussian process | 213_522.65 | 1_032_230.96 | 50_714_268.39 | 70.00 | 771.11 | 3_939 | 6_480 | 20 | 9 | 0.9939 | 0.3370 | 0.5033 |
| ARIMA(2,1,1) | 354_728.53 | 1_286_980.01 | 36_679_628.58 | 3_090 | 11_465.4 | 32_053.25 | 78_437 | 9 | 20 | 0.3460 | 0.6049 | 0.4402 |
| AR(p) | 553_886.69 | 1_286_980.01 | 23_148_609.22 | 3_835.5 | 11_178.38 | 41_818.25 | 78_437 | 13 | 16 | 0.2972 | 0.6902 | 0.4155 |
| LOF | 187_053.1 | 1_040_462.8 | 61_234_992.66 | 45.00 | 48.33 | 97.50 | 110.00 | 23 | 6 | 0.4752 | 0.3008 | 0.3684 |
| Hampel | 194_884.62 | 2_607_213.06 | 63_355_948.02 | 45.00 | 155.92 | 671.95 | 1_151 | 17 | 12 | 0.4452 | 0.1940 | 0.2702 |



#### df2 — доля обучения ρ=80% (test 50 564 точек)


| Модель | RMSE, м | Max(d_i), м | Distance loss, м | Delay median, с | Delay mean, с | Delay p95, с | Delay max, с | On-time episodes | Late/missed episodes | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MPCT | 92_549.46 | 382_289.27 | 1_779_568.51 | 60.00 | 184.00 | 600.00 | 720.00 | 8 | 5 | 0.8237 | 0.8431 | 0.8333 |
| Isolation Forest | 39_152.69 | 92_354.51 | 4_643_322.15 | 110.00 | 110.00 | 110.00 | 110.00 | 12 | 1 | 0.7506 | 0.9346 | 0.8326 |
| One-Class SVM | 222_798.63 | 997_929.19 | 27_062_034.5 | 70.00 | 276.67 | 637.00 | 700.00 | 10 | 3 | 0.7394 | 0.7519 | 0.7456 |
| Фильтр Калмана | 79_001.39 | 459_065.85 | 5_044_268.68 | 120.00 | 1_323.33 | 3_476 | 3_850 | 10 | 3 | 0.8679 | 0.6308 | 0.7306 |
| HMM | 229_762.57 | 1_028_709.53 | 32_568_763.04 | 130.00 | 130.00 | 148.00 | 150.00 | 11 | 2 | 0.7512 | 0.6646 | 0.7053 |
| Robust PCA | 186_251.62 | 6_086_738.96 | 68_552_494.83 | 65.00 | 687.50 | 2_243.5 | 2_620 | 9 | 4 | 0.5410 | 0.8662 | 0.6660 |
| STGVAD | 249_634.96 | 6_086_738.96 | 71_736_070.91 | 40.00 | 323.33 | 1_335 | 1_740 | 7 | 6 | 0.8801 | 0.4979 | 0.6360 |
| DONUT | 202_323.33 | 6_086_738.96 | 46_211_042.37 | 3_170 | 9_098.83 | 28_839.25 | 33_552 | 7 | 6 | 0.9400 | 0.4610 | 0.6186 |
| GP-EKF | 232_297.66 | 1_036_902.9 | 27_147_164.54 | 20.00 | 1_326 | 5_207 | 6_480 | 8 | 5 | 0.8123 | 0.4734 | 0.5982 |
| LSTM-автоэнкодер | 176_675.95 | 6_086_738.96 | 45_411_739.25 | 2_610 | 7_841.86 | 27_896.7 | 33_552 | 6 | 7 | 0.7263 | 0.4691 | 0.5700 |
| TCN | 257_162.32 | 6_086_738.96 | 66_214_884.79 | 20.00 | 1_727.89 | 9_090.6 | 15_011 | 4 | 9 | 0.9102 | 0.3279 | 0.4821 |
| AR(p) | 652_061.92 | 6_086_738.96 | 38_499_286.04 | — | — | — | — | 13 | 0 | 0.3097 | 0.9997 | 0.4729 |
| ARIMA(2,1,1) | 652_061.92 | 6_086_738.96 | 38_499_286.04 | — | — | — | — | 13 | 0 | 0.3097 | 0.9997 | 0.4729 |
| LOF | 211_172.71 | 1_036_902.9 | 40_176_324.99 | 10.00 | 36.67 | 91.00 | 100.00 | 10 | 3 | 0.4655 | 0.4095 | 0.4357 |
| Hampel | 235_966.42 | 2_606_113.66 | 45_913_559.87 | 40.00 | 81.67 | 240.00 | 280.00 | 7 | 6 | 0.4783 | 0.1763 | 0.2576 |
| Gaussian process | 257_462.88 | 2_606_113.66 | 46_885_161.12 | 280.00 | 2_176.67 | 6_335 | 6_480 | 7 | 6 | 0.9848 | 0.0496 | 0.0944 |



### Сортировка: Distance loss


#### df1 — доля обучения ρ=60% (test 86 263 точек)


| Модель | RMSE, м | Max(d_i), м | Distance loss, м | Delay median, с | Delay mean, с | Delay p95, с | Delay max, с | On-time episodes | Late/missed episodes | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MPCT | 73_201.74 | 380_633.64 | 2_764_635.44 | 10.00 | 91.82 | 420.00 | 720.00 | 18 | 11 | 0.8527 | 0.8371 | 0.8448 |
| Фильтр Калмана | 63_605.52 | 456_336.43 | 8_104_878.95 | 65.00 | 995.00 | 3_290.5 | 3_850 | 25 | 4 | 0.8095 | 0.7094 | 0.7562 |
| Isolation Forest | 44_337.01 | 472_436.06 | 11_422_169.28 | 110.00 | 110.00 | 110.00 | 110.00 | 28 | 1 | 0.6827 | 0.9025 | 0.7774 |
| GP-EKF | 141_435 | 1_032_210.33 | 22_281_191.9 | 100.00 | 846.00 | 3_223 | 4_000 | 24 | 5 | 0.6140 | 0.5728 | 0.5927 |
| AR(p) | 553_886.69 | 1_286_980.01 | 23_148_609.22 | 3_835.5 | 11_178.38 | 41_818.25 | 78_437 | 13 | 16 | 0.2972 | 0.6902 | 0.4155 |
| One-Class SVM | 135_586.87 | 927_706.26 | 24_177_277.58 | 60.00 | 86.00 | 222.00 | 260.00 | 24 | 5 | 0.7611 | 0.8389 | 0.7981 |
| HMM | 173_539.13 | 1_032_210.33 | 35_757_396.53 | 10.00 | 10.00 | 19.00 | 20.00 | 26 | 3 | 0.6544 | 0.7816 | 0.7124 |
| ARIMA(2,1,1) | 354_728.53 | 1_286_980.01 | 36_679_628.58 | 3_090 | 11_465.4 | 32_053.25 | 78_437 | 9 | 20 | 0.3460 | 0.6049 | 0.4402 |
| Gaussian process | 213_522.65 | 1_032_230.96 | 50_714_268.39 | 70.00 | 771.11 | 3_939 | 6_480 | 20 | 9 | 0.9939 | 0.3370 | 0.5033 |
| LOF | 187_053.1 | 1_040_462.8 | 61_234_992.66 | 45.00 | 48.33 | 97.50 | 110.00 | 23 | 6 | 0.4752 | 0.3008 | 0.3684 |
| Hampel | 194_884.62 | 2_607_213.06 | 63_355_948.02 | 45.00 | 155.92 | 671.95 | 1_151 | 17 | 12 | 0.4452 | 0.1940 | 0.2702 |
| LSTM-автоэнкодер | 171_426.13 | 6_380_999.81 | 86_266_364.98 | 930.50 | 6_863 | 29_140 | 33_552 | 17 | 12 | 0.8709 | 0.5815 | 0.6974 |
| DONUT | 192_551.17 | 6_380_999.81 | 89_189_301.72 | 770.00 | 6_358.15 | 28_739 | 33_552 | 16 | 13 | 0.9553 | 0.5671 | 0.7117 |
| TCN | 224_256.21 | 6_380_999.81 | 116_632_211.55 | 170.00 | 2_519.59 | 17_159.2 | 25_752 | 12 | 17 | 0.9078 | 0.4306 | 0.5841 |
| Robust PCA | 158_451.27 | 6_380_999.81 | 117_428_236.07 | 30.00 | 3_217.75 | 16_626.3 | 25_552 | 21 | 8 | 0.4646 | 0.8651 | 0.6045 |
| STGVAD | 213_800.55 | 6_380_999.81 | 124_870_896.61 | 45.00 | 1_708.5 | 5_796.3 | 25_552 | 11 | 18 | 0.7970 | 0.5111 | 0.6228 |



#### df2 — доля обучения ρ=80% (test 50 564 точек)


| Модель | RMSE, м | Max(d_i), м | Distance loss, м | Delay median, с | Delay mean, с | Delay p95, с | Delay max, с | On-time episodes | Late/missed episodes | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MPCT | 92_549.46 | 382_289.27 | 1_779_568.51 | 60.00 | 184.00 | 600.00 | 720.00 | 8 | 5 | 0.8237 | 0.8431 | 0.8333 |
| Isolation Forest | 39_152.69 | 92_354.51 | 4_643_322.15 | 110.00 | 110.00 | 110.00 | 110.00 | 12 | 1 | 0.7506 | 0.9346 | 0.8326 |
| Фильтр Калмана | 79_001.39 | 459_065.85 | 5_044_268.68 | 120.00 | 1_323.33 | 3_476 | 3_850 | 10 | 3 | 0.8679 | 0.6308 | 0.7306 |
| One-Class SVM | 222_798.63 | 997_929.19 | 27_062_034.5 | 70.00 | 276.67 | 637.00 | 700.00 | 10 | 3 | 0.7394 | 0.7519 | 0.7456 |
| GP-EKF | 232_297.66 | 1_036_902.9 | 27_147_164.54 | 20.00 | 1_326 | 5_207 | 6_480 | 8 | 5 | 0.8123 | 0.4734 | 0.5982 |
| HMM | 229_762.57 | 1_028_709.53 | 32_568_763.04 | 130.00 | 130.00 | 148.00 | 150.00 | 11 | 2 | 0.7512 | 0.6646 | 0.7053 |
| AR(p) | 652_061.92 | 6_086_738.96 | 38_499_286.04 | — | — | — | — | 13 | 0 | 0.3097 | 0.9997 | 0.4729 |
| ARIMA(2,1,1) | 652_061.92 | 6_086_738.96 | 38_499_286.04 | — | — | — | — | 13 | 0 | 0.3097 | 0.9997 | 0.4729 |
| LOF | 211_172.71 | 1_036_902.9 | 40_176_324.99 | 10.00 | 36.67 | 91.00 | 100.00 | 10 | 3 | 0.4655 | 0.4095 | 0.4357 |
| LSTM-автоэнкодер | 176_675.95 | 6_086_738.96 | 45_411_739.25 | 2_610 | 7_841.86 | 27_896.7 | 33_552 | 6 | 7 | 0.7263 | 0.4691 | 0.5700 |
| Hampel | 235_966.42 | 2_606_113.66 | 45_913_559.87 | 40.00 | 81.67 | 240.00 | 280.00 | 7 | 6 | 0.4783 | 0.1763 | 0.2576 |
| DONUT | 202_323.33 | 6_086_738.96 | 46_211_042.37 | 3_170 | 9_098.83 | 28_839.25 | 33_552 | 7 | 6 | 0.9400 | 0.4610 | 0.6186 |
| Gaussian process | 257_462.88 | 2_606_113.66 | 46_885_161.12 | 280.00 | 2_176.67 | 6_335 | 6_480 | 7 | 6 | 0.9848 | 0.0496 | 0.0944 |
| TCN | 257_162.32 | 6_086_738.96 | 66_214_884.79 | 20.00 | 1_727.89 | 9_090.6 | 15_011 | 4 | 9 | 0.9102 | 0.3279 | 0.4821 |
| Robust PCA | 186_251.62 | 6_086_738.96 | 68_552_494.83 | 65.00 | 687.50 | 2_243.5 | 2_620 | 9 | 4 | 0.5410 | 0.8662 | 0.6660 |
| STGVAD | 249_634.96 | 6_086_738.96 | 71_736_070.91 | 40.00 | 323.33 | 1_335 | 1_740 | 7 | 6 | 0.8801 | 0.4979 | 0.6360 |


## Выводы

### Устройство данных

Исходный CSV: 419 453 строк. После отбора координат, статусов move и anomaly, сортировки по времени и обрезки по крайним точкам move остаётся 252 818 наблюдений.

| Сценарий | $\rho$ | Обучение | Доля anomaly на обучении | Тест | Доля anomaly на тесте |
| --- | --- | --- | --- | --- | --- |
| df1 | 60% | 166 555 | 69.5% | 86 263 | 27.5% |
| df2 | 80% | 202 254 | 61.2% | 50 564 | 31.0% |

Тест df2 — хвост того же ряда, что и тест df1. Согласие рангов на двух $\rho$ говорит об устойчивости вдоль трека.

### Тестирование

Ранги по $F_1$ (убывание) и по distance loss (возрастание). DL указан в миллионах метров, чтобы порядки читались рядом.

| Место | df1 по $F_1$ | $F_1$ | DL, млн м | df2 по $F_1$ | $F_1$ | DL, млн м |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | MPCT | 0.845 | 2.76 | MPCT | 0.833 | 1.78 |
| 2 | One-Class SVM | 0.798 | 24.2 | Isolation Forest | 0.833 | 4.64 |
| 3 | Isolation Forest | 0.777 | 11.4 | One-Class SVM | 0.746 | 27.1 |
| 4 | Фильтр Калмана | 0.756 | 8.10 | Фильтр Калмана | 0.731 | 5.04 |
| 5 | HMM | 0.712 | 35.8 | HMM | 0.705 | 32.6 |
| 6 | DONUT | 0.712 | 89.2 | Robust PCA | 0.666 | 68.6 |
| 16 | Hampel | 0.270 | 63.4 | Gaussian process | 0.094 | 46.9 |

По distance loss первая тройка на обоих сценариях одна и та же: MPCT, фильтр Калмана, Isolation Forest. На df1 следом GP-EKF (22.3 млн м) при $F_1$ только 0.593. На df2 четвёртое-пятое места делят One-Class SVM и GP-EKF (около 27 млн м).

Isolation Forest даёт лучший RMSE на обоих тестах: 44.3 км (df1) и 39.2 км (df2). У MPCT RMSE 73.2 и 92.5 км при лучшем DL. Калман — 63.6 и 79.0 км. Линейные модели на df2 выходят на RMSE 652 км при том же Max$(d_i)$, что и у нейросетей.

Задержка отделяет «попал в метку» от «успел на эпизод». HMM на df1 срабатывает почти сразу: медиана 10 с, 95-й перцентиль 19 с, 26 on-time против 3 late/missed. Isolation Forest стабилен: медиана ровно 110 с на обоих сценариях, late/missed — 1 эпизод из 29 и из 13. MPCT точнее по $F_1$, но опаздывает чаще: 11 и 5 late/missed, хвост задержки до 720 с. У Калмана медиана терпимая (65 и 120 с), зато среднее около 1000–1300 с и 95-й перцентиль в тысячи секунд: часть эпизодов находится очень поздно. LSTM и DONUT опаздывают на часы (медианы от 770 с до 3170 с, максимум 33 552 с).

Сдвиг $\rho$ с 60% на 80% почти не двигает лидеров и ломает два метода. $F_1$ MPCT 0.845 $\to$ 0.833, Isolation Forest 0.777 $\to$ 0.833, Калман 0.756 $\to$ 0.731, HMM 0.712 $\to$ 0.705. Gaussian process 0.503 $\to$ 0.094 при precision 0.985 и recall 0.050: детектор становится почти глухим. AR($p$) и ARIMA на df2 совпадают до последней цифры (precision 0.310, recall 0.9997, $F_1$ 0.473, задержки не определены). Доля anomaly на этом тесте — 31.0%. Совпадение precision с этой долей при полном recall означает, что обе модели пометили аномалией практически каждую точку теста. На df1 они ещё различаются ($F_1$ 0.440 и 0.416) и обе имеют задержку в часы.

### Сильные и слабые стороны по группам

**MPCT.** Лучший $F_1$ и лучший DL на обоих разрезах, precision и recall одновременно выше 0.82. Метод прямо кодирует то, чем размеченный anomaly, судя по результату, и отличается от move: скачок скорости относительно последнего доверенного положения и изломанная форма короткого окна. Ему не мешает, что на обучении anomaly — большинство: оцениваются два порога, а не плотность смеси. Слабые места — эвристический якорь вместо глобально оптимальной подпоследовательности, фиксированная геометрия окна и более слабый RMSE, чем у леса. Часть эпизодов он пропускает или берет с задержкой до 12 минут. Имеет смысл как основной физический детектор или как ограничитель для любой статистической маски: точка, которую MPCT отвергает, не должна молча оставаться в ломаной.

**Isolation Forest.** Второй устойчивый результат и лучшая геометрия точек (RMSE, на df2 ещё и Max$(d_i)$ 92 км против сотен у остальных). Recall 0.90 и 0.93, precision 0.68 и 0.75: лес скорее помечает лишнее, чем пропускает. Задержка короткая и одинаковая на обоих хвостах. Лес видит смесь классов на обучении и всё равно отделяет редкий кинематический режим на тесте, где anomaly уже не большинство. Ограничение: нет модели движения и нет причинности внутри дерева; признак формы и шага не отличает «судно повернуло штатно, но редко» от поломки трека. DL хуже Калмана на df1 (11.4 против 8.1 млн м) при лучшем $F_1$, то есть лишние ложные anomaly режут ломаную не так болезненно, как пропуски, но и не дотягивают до MPCT.

**Фильтр Калмана.** Лучший компромисс среди моделей с состоянием: $F_1$ около 0.73–0.76, DL сразу за MPCT, precision на df2 равен 0.87 при recall 0.63. Постоянная скорость хорошо описывает куски реального хода и плохо — долгий аномальный режим, пока инновация не накопится: отсюда тяжёлый хвост задержки. Сброс на паузе 180 с честно обрабатывает разрыв связи, но обнуление скорости после паузы само может дать всплеск score. Улучшение здесь предметное: несколько режимов движения (IMM), отдельный режим «стоп», ограничение на принятие точки по правилу MPCT.

**One-Class SVM.** Высокий $F_1$ за счёт баланса precision и recall, на df1 задержка умеренная (медиана 60 с, 24 on-time из 29). На df2 $F_1$ проседает до 0.75, Max$(d_i)$ около 1000 км, DL около 27 млн м. Граница RBF на подвыборке 4000 точек move чувствительна к тому, какие штатные манёвры попали в опору. Фиксированный порог $f<0$ не участвует в полосе MCC, поэтому операционный DL при выборе $\nu$ и длины ядра может расходиться с тем, как выбирался $\tau$ у соседей.

**HMM.** Единственная модель с явной гипотезой «аномалия — другой марковский режим». Задержка на df1 лучшая в опыте, $F_1$ стабильно около 0.71. Precision 0.65 и 0.75 при recall 0.78 и 0.66: путаница симметричнее, чем у нейросетей. Геометрия слабая (RMSE 174 и 230 км, DL 36 и 33 млн м): метка эпизода ставится вовремя, но принятые и отвергнутые точки всё ещё плохо собирают длину.

**GP и GP-EKF.** Процесс на шаге слишком осторожен: precision 0.99 на обоих тестах, recall 0.34 и затем 0.05. Длинная корреляция по времени и обучение только на move дают узкую трубку типичного шага; всё, что в неё попало, принимается, остальное почти не встречается на тесте df2. GP-EKF честнее как фильтр: $F_1$ около 0.59 на обоих разрезах, DL лучше голого GP (22 и 27 млн м), recall всё ещё около 0.5. Четыре независимых скалярных GP, не больше 160 опор и сетка гиперпараметров по правдоподобию не компенсируют отсутствие связи координат и бедность опоры.

**AR и ARIMA.** На всём обучении, где anomaly составляет 60–70%, коэффициенты подстраиваются под сломанные шаги. На длинном тесте оба прогноза перестают отличать режимы и сваливаются в одну маску «всё anomaly». Одинаковые цифры на df2 — симптом вырождения порога, а не совпадения двух теорий. Задержки в часы на df1 показывают, что даже там одношаговая ошибка уровня координат реагирует слишком поздно. Линейный прогноз уровня ENU для этой разметки слабый.

**Hampel и LOF.** $F_1$ 0.27 и 0.26 у Hampel, 0.37 и 0.44 у LOF, recall Hampel около 0.19. Локальная плотность и медиана скорости ловят изолированные скачки. Размеченная anomaly на этом ряду — протяжённый режим: внутри режима точка похожа на соседей, и LOF/Hampel её отпускают. Задержка у пойманных эпизодов при этом короткая (у LOF медиана 45 и 10 с).

**Robust PCA.** На df2 $F_1$ 0.67 выше всех оконных сетей, recall 0.87, но precision 0.54 и DL 69 млн м. Разреженная компонента окон не совпадает с геометрией принятой ломаной: низкий ранг забирает плавный ход, а в $S$ остаётся и штатный манёвр, и кусок anomaly. Max$(d_i)$ снова на общем потолке тысяч километров.

**LSTM, TCN, DONUT, STGVAD.** Общий портрет: precision высокий (часто 0.87–0.96), recall 0.33–0.58, DL на порядок хуже Калмана, задержка эпизода — тысячи секунд, кроме более быстрого STGVAD (медиана 45 и 40 с) ценой худшего DL в опыте. Окно относительно собственной первой точки хорошо видит «это окно не похоже на штатные окна» и плохо покрывает длинный аномальный участок: max по окнам зажигает точку только если какое-то окно целиком странное, а короткие окна внутри долгого плохого режима могут выглядеть внутренне согласованными. Обучение только на move при доле anomaly на обучении около двух третей ещё и оставляет мало штатных окон.

### Выводы
MPCT лучше всех совпадает с разметкой и меньше всех портит длину ломаной. Isolation Forest лучше всех держит RMSE и почти не пропускает эпизоды, с устойчивой задержкой около двух минут. Фильтр Калмана — лучший онлайн-фильтр с состоянием: чуть слабее по $F_1$, заметно сильнее по DL, чем признаковые и оконные альтернативы, с тяжёлым хвостом задержки. HMM уместен, если есть оба класса, и даёт самую быструю реакцию ценой геометрии. One-Class SVM силён как классификатор признаков и слаб как источник принятой траектории.

Линейный прогноз уровней по смеси классов на длинном тесте вырождается в метку «всё anomaly». Гауссовский процесс шага вырождается в обратную сторону: почти ничего не объявляет аномалией. Локальные выбросовые детекторы не видят протяжённый режим. Оконные нейросети в текущей адаптации устойчиво дают высокую precision, низкий recall, многочасовую задержку и хорды в тысячи километров.